# Change the Test Path Only


In [ ]:
# ================================================================
# EDITABLE KAGGLE INPUT PATHS
# Update only these attached-asset paths if Kaggle assigns new versions.
# Do NOT change TEST_PATH: the host replaces it with the held-out test set.
# ================================================================

#####
TEST_PATH = "/kaggle/input/competitions/bengali-hallucination/test set.csv"
#####

# Final hallucination judge and its offline vLLM wheel bundle.
MODEL_PATH = "/kaggle/input/models/awooooo/qwen3-5-35b-a3b-gptq-int4/other/gptq-int4/1"
VLLM_WHEELS_DIR = "/kaggle/input/datasets/tukotanzwoo/vllm-wheels"

# Single deterministic cascade classifier.py.
CLASSIFIER_PATH = "/kaggle/input/datasets/supriopaul/classifier/classifier.py"

# Retrieval index used after deterministic category selection.
RAG_INDEX_DIR = "/kaggle/input/datasets/afhamadian/kb-v4-withoutnews/rag_index"

# Offline BGE models already attached as Kaggle Models.
BGE_M3_PATH = "/kaggle/input/models/yethukmutt/bge-m3/transformers/m3/1/bge-m3"
RERANKER_PATH = "/kaggle/input/models/andreasbis/baai-bge-reranker-v2-m3/transformers/default/1"

# Dataset containing the downloaded FlagEmbedding and faiss-cpu wheels.
RAG_WHEELS_DIR = "/kaggle/input/datasets/supriopaul/rag-wheels/rag_wheels"

# Curated idiom/meaning archive used by the unchanged Qwen pipeline.
LEXICAL_ARCHIVE_PATH = "/kaggle/input/datasets/sakhadib/bagdhara-bangla-idioms-dataset"

# Runtime outputs. These are generated by this notebook.
SEPARATED_DIR = "/kaggle/working/separated"
RELATION_RAG_OUTPUT_DIR = "/kaggle/working/relation_rag"
RAG_CONTEXT_PATH = f"{RELATION_RAG_OUTPUT_DIR}/rag_output_top3_classified_0.75.csv"
OUTPUT_DIR = "/kaggle/working"

INSTALL_RAG_FROM_WHEELS = True
INSTALL_VLLM_FROM_WHEELS = True


In [ ]:
# Generate the four classifier CSVs from the official test set.
# The classifier runs in a subprocess to keep its implementation outside this notebook.

import subprocess
import sys
from pathlib import Path


def find_classifier_script(configured_path):
    configured_path = Path(configured_path)
    if configured_path.is_file():
        return configured_path
    if not configured_path.exists():
        raise FileNotFoundError(f"Classifier path not found: {configured_path}")
    matches = sorted(configured_path.rglob("classifier.py"))
    if not matches:
        raise FileNotFoundError(f"classifier.py not found under: {configured_path}")
    return matches[0]


classifier_script = find_classifier_script(CLASSIFIER_PATH)

Path(SEPARATED_DIR).mkdir(parents=True, exist_ok=True)
command = [
    sys.executable,
    str(classifier_script),
    "--input",
    TEST_PATH,
    "--output-dir",
    SEPARATED_DIR,
]
print("Running deterministic cascade:", " ".join(command))
subprocess.run(command, check=True)

required_outputs = [
    "math.csv",
    "only_bangla.csv",
    "null_only_gk.csv",
    "non_null_only_gk.csv",
]
missing_outputs = [name for name in required_outputs if not (Path(SEPARATED_DIR) / name).is_file()]
if missing_outputs:
    raise RuntimeError(f"Classifier subprocess did not produce: {missing_outputs}")
print("Classifier outputs ready in:", SEPARATED_DIR)


## Category-gated live retrieval

The classifier has already produced all four buckets. The next cells retrieve
only rows whose category needs external evidence: GK/context/eligible grammar,
plus a specialized relation strategy for synonym/antonym. Math and direct
archive lexical rows are skipped. Retrieval runs in a child process and exits
before vLLM loads.


In [ ]:
# Offline retrieval dependency check/install.
# Core Kaggle packages are left untouched. Only missing FlagEmbedding/faiss-cpu
# are installed from the attached wheel dataset, with dependencies disabled.

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def install_rag_dependencies_offline(wheels_root):
    packages = []
    if importlib.util.find_spec("FlagEmbedding") is None:
        packages.append("FlagEmbedding")
    if importlib.util.find_spec("faiss") is None:
        packages.append("faiss-cpu")
    if not packages:
        print("Offline RAG packages already available.")
        return

    wheels_root = Path(wheels_root)
    if not wheels_root.exists():
        raise FileNotFoundError(
            f"RAG wheel directory not found: {wheels_root}. "
            "Update RAG_WHEELS_DIR in the editable path cell."
        )
    wheel_files = sorted(wheels_root.rglob("*.whl"))
    if not wheel_files:
        raise FileNotFoundError(f"No wheel files found under: {wheels_root}")
    wheel_dirs = sorted({str(path.parent) for path in wheel_files})
    command = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
    for directory in wheel_dirs:
        command.extend(["--find-links", directory])
    command.extend(packages)
    print("Installing offline RAG packages:", packages)
    subprocess.check_call(command)


if INSTALL_RAG_FROM_WHEELS:
    install_rag_dependencies_offline(RAG_WHEELS_DIR)

# Verify imports in a child process so no Torch/CUDA state is retained here.
subprocess.check_call(
    [
        sys.executable,
        "-c",
        (
            "import faiss, scipy; "
            "from FlagEmbedding import BGEM3FlagModel; "
            "from sentence_transformers import CrossEncoder; "
            "print('Offline RAG dependency check: PASS')"
        ),
    ]
)


In [ ]:
%%writefile /kaggle/working/offline_relation_rag_stage.py
"""Offline category-gated retrieval stage for the final Kaggle notebook.

This stage runs only after the deterministic classifier has produced its four
CSV buckets. It performs ordinary retrieval for GK/context/eligible grammar
rows and specialized multi-query retrieval for synonym/antonym prompts. Math
and archive-handled direct lexical rows are excluded. It never merges the
original question context into retrieved evidence.
"""

import argparse
import gc
import re
import time
import unicodedata
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from FlagEmbedding import BGEM3FlagModel
from sentence_transformers import CrossEncoder


RELATION_HINTS = {
    "antonym": ["বিপরীতার্থক", "বিপরীত শব্দ", "বিপরীত অর্থ"],
    "synonym": ["সমার্থক", "প্রতিশব্দ"],
}
RELATION_MARKERS = {
    "antonym": ["বিপরীতার্থক", "বিপরীত শব্দ", "বিপরীত", "antonym"],
    "synonym": ["সমার্থক", "প্রতিশব্দ", "synonym"],
}
CLASSIFIER_FILES = [
    "math.csv",
    "only_bangla.csv",
    "null_only_gk.csv",
    "non_null_only_gk.csv",
]
DIRECT_LEXICAL_HINTS = [
    "শব্দার্থ",
    "শাব্দিক অর্থ",
    "ভাবার্থ",
    "বাগধারা",
    "বাগধারটির",
    "প্রবাদ",
]
DIRECT_LEXICAL_PATTERNS = [
    r"(?:শব্দ(?:টি|টির)?|শব্দের)\s+(?:শুদ্ধ\s+)?(?:বাংলা\s+)?(?:অর্থ|মানে)",
    r"(?:কথা(?:টি|টির)?|কথার)\s+(?:বাংলা\s+)?(?:অর্থ|মানে)",
    r"[\"'][^\"']{1,100}[\"']\s*(?:[-–—]\s*)?(?:এর\s*)?(?:অর্থ|মানে|ভাবার্থ)",
    r"^.{1,80}?\s+এর\s+(?:শাব্দিক\s+)?(?:অর্থ|মানে|ভাবার্থ)\s*(?:কী|কি|বলতে)",
]


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--test-path", required=True)
    parser.add_argument("--separated-dir", required=True)
    parser.add_argument("--index-dir", required=True)
    parser.add_argument("--embed-model", required=True)
    parser.add_argument("--reranker-model", required=True)
    parser.add_argument("--output-dir", required=True)
    parser.add_argument("--nprobe", type=int, default=4096)
    parser.add_argument("--top-k-retrieve", type=int, default=30)
    parser.add_argument("--variant-top-k", type=int, default=10)
    parser.add_argument("--final-top-k", type=int, default=4)
    parser.add_argument("--score-threshold", type=float, default=0.75)
    parser.add_argument("--alpha-dense", type=float, default=0.7)
    parser.add_argument("--encode-batch-size", type=int, default=32)
    parser.add_argument("--rerank-batch-size", type=int, default=64)
    return parser.parse_args()


def normalize_text(value):
    if pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKC", str(value)).strip()
    text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    text = re.sub(r"[\u200c\u200d]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def contains_whole_term(text, term):
    text = normalize_text(text).lower()
    term = normalize_text(term).lower().strip(" ?।:-–—\"'")
    if not term:
        return False
    boundary_chars = r"\w\u0980-\u09ff"
    pattern = rf"(?<![{boundary_chars}]){re.escape(term)}(?![{boundary_chars}])"
    return re.search(pattern, text, flags=re.IGNORECASE) is not None


def relation_type(prompt):
    text = normalize_text(prompt).lower()
    for kind in ("antonym", "synonym"):
        if any(hint in text for hint in RELATION_HINTS[kind]):
            return kind
    return ""


def is_direct_lexical_prompt(prompt):
    """Rows handled by the curated lexical/idiom archive, not general RAG."""
    text = normalize_text(prompt)
    if relation_type(text):
        return False
    if any(hint in text for hint in DIRECT_LEXICAL_HINTS):
        return True
    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in DIRECT_LEXICAL_PATTERNS)


def valid_target(term):
    term = normalize_text(term).strip(" ?।:-–—\"'")
    return 1 < len(term) <= 100 and len(term.split()) <= 12 and "?" not in term


def extract_relation_target(prompt):
    text = normalize_text(prompt)
    if not relation_type(text):
        return ""

    quoted = re.search(r'[\"\']([^\"\']{1,100})[\"\']', text)
    if quoted and valid_target(quoted.group(1)):
        return quoted.group(1).strip(" ?।:-–—\"'")

    patterns = [
        r"^(?:এখানে\s+)?(.{1,80}?)[\"']?\s+(?:শব্দ(?:টি|টির)?|শব্দের)\s+(?:শুদ্ধ\s+)?(?:সমার্থক(?:বাচক)?|প্রতিশব্দ|বিপরীতার্থক|বিপরীত\s+(?:শব্দ|অর্থ))",
        r"^(?:এখানে\s+)?(.{1,80}?)[\"']?\s+এর\s+(?:সমার্থক(?:বাচক)?|প্রতিশব্দ|বিপরীতার্থক|বিপরীত\s+(?:শব্দ|অর্থ))",
        r"^(?:এখানে\s+)?(.{1,80}?)[\"']?\s+(?:সমার্থক(?:বাচক)?|প্রতিশব্দ|বিপরীতার্থক|বিপরীত\s+(?:শব্দ|অর্থ))",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match and valid_target(match.group(1)):
            return match.group(1).strip(" ?।:-–—\"'")
    return ""


def make_query_spec(row):
    kind = relation_type(row.prompt_bn)
    target = extract_relation_target(row.prompt_bn)
    candidate = normalize_text(row.response_bn)
    if not kind or not target or not candidate:
        return None

    relation_bn = "বিপরীতার্থক" if kind == "antonym" else "সমার্থক"
    canonical = (
        f'মূল শব্দ: "{target}"। প্রার্থী উত্তর: "{candidate}"। '
        f"যাচাই করো শব্দ দুটি সত্যিই {relation_bn} সম্পর্কযুক্ত কি না।"
    )
    variants = [
        normalize_text(row.prompt_bn),
        f'"{target}" শব্দের {relation_bn} শব্দ কী কী?',
        canonical,
        f'"{candidate}" এবং "{target}" শব্দ দুটি কি {relation_bn}?',
    ]
    return {
        "id": row.id,
        "relation_type": kind,
        "target_word": target,
        "candidate_word": candidate,
        "prompt_bn": row.prompt_bn,
        "response_bn": row.response_bn,
        "canonical_query": canonical,
        "query_variants": list(dict.fromkeys(query for query in variants if query)),
    }


def split_qa_document(document):
    text = normalize_text(document)
    marker = re.search(r"(?:^|\s)(?:Ans(?:wer)?|উত্তর)\s*[:：-]\s*", text, flags=re.IGNORECASE)
    if marker is None:
        return text, ""
    return text[: marker.start()].strip(), text[marker.end() :].strip()


def has_relation_marker(text, kind):
    normalized = normalize_text(text).lower()
    return any(marker.lower() in normalized for marker in RELATION_MARKERS[kind])


def answer_is_negative(answer):
    normalized = normalize_text(answer).lower()
    return (
        contains_whole_term(normalized, "না")
        or "নয়" in normalized
        or "নয়" in normalized
        or contains_whole_term(normalized, "not")
        or "সম্পর্কযুক্ত নয়" in normalized
        or "সম্পর্কযুক্ত নয়" in normalized
    )


def relation_flags(document, spec):
    question, answer = split_qa_document(document)
    kind = spec["relation_type"]
    target = spec["target_word"]
    candidate = spec["candidate_word"]

    target_hit = contains_whole_term(document, target)
    candidate_hit = contains_whole_term(document, candidate)
    marker_hit = has_relation_marker(document, kind)
    question_has_target = contains_whole_term(question, target)
    question_has_candidate = contains_whole_term(question, candidate)
    question_has_marker = has_relation_marker(question, kind)
    answer_has_target = contains_whole_term(answer, target)
    answer_has_candidate = contains_whole_term(answer, candidate)
    positive_answer = bool(answer) and not answer_is_negative(answer)
    pair_question = contains_whole_term(question, "ও") or contains_whole_term(question, "এবং")

    direct_relation = positive_answer and question_has_marker and (
        (not pair_question and question_has_target and answer_has_candidate)
        or (not pair_question and question_has_candidate and answer_has_target)
        or (
            pair_question
            and question_has_target
            and question_has_candidate
            and (
                contains_whole_term(answer, "হ্যাঁ")
                or contains_whole_term(answer, "হ্যা")
                or contains_whole_term(answer, "yes")
                or "পরস্পরের" in normalize_text(answer)
            )
        )
    )
    cooccurrence_pair = target_hit and candidate_hit and marker_hit
    return {
        "target_hit": target_hit,
        "candidate_hit": candidate_hit,
        "relation_marker_hit": marker_hit,
        "target_relation_hit": question_has_target and question_has_marker,
        "candidate_relation_hit": question_has_candidate and question_has_marker,
        "cooccurrence_pair_hit": cooccurrence_pair,
        "cooccurrence_only": cooccurrence_pair and not direct_relation,
        "direct_relation_assertion": direct_relation,
    }


def minmax(values):
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0 or values.max() - values.min() < 1e-9:
        return np.zeros_like(values)
    return (values - values.min()) / (values.max() - values.min())


def id_key(value):
    try:
        number = float(value)
        if number.is_integer():
            return str(int(number))
    except (TypeError, ValueError):
        pass
    return normalize_text(value)


def load_classified_rag_rows(test_path, separated_dir):
    separated_dir = Path(separated_dir)
    relation_parts = []
    general_parts = []
    stats = {}
    for filename in CLASSIFIER_FILES:
        path = separated_dir / filename
        if not path.is_file():
            raise FileNotFoundError(f"Missing classifier output: {path}")
        frame = pd.read_csv(path)
        required = {"id", "prompt_bn", "response_bn"}
        missing = required - set(frame.columns)
        if missing:
            raise ValueError(f"{path} is missing columns: {sorted(missing)}")
        relation_mask = frame["prompt_bn"].apply(lambda value: bool(relation_type(value)))
        direct_lexical_mask = frame["prompt_bn"].apply(is_direct_lexical_prompt)

        relation = frame.loc[relation_mask, ["id", "prompt_bn", "response_bn"]].copy()
        relation["classifier_source"] = filename
        relation_parts.append(relation)

        if filename in {"null_only_gk.csv", "non_null_only_gk.csv"}:
            general_mask = ~relation_mask
        elif filename == "only_bangla.csv":
            general_mask = ~relation_mask & ~direct_lexical_mask
        else:
            general_mask = pd.Series(False, index=frame.index)
        general = frame.loc[general_mask, ["id", "prompt_bn", "response_bn"]].copy()
        general["classifier_source"] = filename
        general_parts.append(general)

        stats[filename] = {
            "total": len(frame),
            "relation_rag": int(relation_mask.sum()),
            "general_rag": int(general_mask.sum()),
            "archive_only": int((direct_lexical_mask & ~relation_mask).sum()),
        }

    relation_ids = pd.concat(relation_parts, ignore_index=True)
    general_ids = pd.concat(general_parts, ignore_index=True)
    selected_ids = pd.concat(
        [
            relation_ids.assign(retrieval_mode="relation"),
            general_ids.assign(retrieval_mode="general"),
        ],
        ignore_index=True,
    )
    if selected_ids["id"].map(id_key).duplicated().any():
        duplicates = selected_ids.loc[
            selected_ids["id"].map(id_key).duplicated(keep=False), "id"
        ].tolist()
        raise ValueError(f"RAG IDs occur in multiple modes/buckets: {duplicates[:20]}")

    test_df = pd.read_csv(test_path)
    required = {"id", "context", "prompt_bn", "response_bn"}
    missing = required - set(test_df.columns)
    if missing:
        raise ValueError(f"Official test file is missing columns: {sorted(missing)}")
    metadata_by_id = selected_ids.set_index(selected_ids["id"].map(id_key))[
        ["classifier_source", "retrieval_mode"]
    ].to_dict("index")
    test_df["id_key"] = test_df["id"].map(id_key)
    selected = test_df[test_df["id_key"].isin(metadata_by_id)].copy()
    selected["classifier_source"] = selected["id_key"].map(
        lambda key: metadata_by_id[key]["classifier_source"]
    )
    selected["retrieval_mode"] = selected["id_key"].map(
        lambda key: metadata_by_id[key]["retrieval_mode"]
    )

    if len(selected) != len(selected_ids):
        raise ValueError(
            f"Classifier/test RAG mismatch: classifier={len(selected_ids)}, test={len(selected)}"
        )
    relation_rows = selected[selected["retrieval_mode"] == "relation"].drop(columns="id_key")
    general_rows = selected[selected["retrieval_mode"] == "general"].drop(columns="id_key")
    print("RAG selection after deterministic classification:")
    for filename, values in stats.items():
        print(f"  {filename}: {values}")
    print("  specialized relation rows:", len(relation_rows))
    print("  ordinary retrieval rows:", len(general_rows))
    print("  total live retrieval rows:", len(selected))
    return relation_rows, general_rows


class RelationRetriever:
    def __init__(self, args):
        self.args = args
        self.index_dir = Path(args.index_dir)
        required = [
            self.index_dir / "faiss_ivf_sq8.index",
            self.index_dir / "sparse_all.npz",
            self.index_dir / "doc_texts.txt",
        ]
        missing = [str(path) for path in required if not path.is_file()]
        if missing:
            raise FileNotFoundError(f"Missing RAG index assets: {missing}")

        self.index = faiss.read_index(str(required[0]))
        self.index.nprobe = args.nprobe
        self.sparse_all = sp.load_npz(required[1])
        with open(required[2], encoding="utf-8", newline="\n") as handle:
            self.doc_texts = [line.rstrip("\n").replace("\\n", "\n") for line in handle]
        if self.index.ntotal != len(self.doc_texts) or self.sparse_all.shape[0] != len(self.doc_texts):
            raise ValueError(
                "RAG assets are misaligned: "
                f"faiss={self.index.ntotal}, sparse={self.sparse_all.shape[0]}, "
                f"docs={len(self.doc_texts)}"
            )

        self.ngpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
        self.device = "cuda" if self.ngpu else "cpu"
        self.embed_model = BGEM3FlagModel(
            args.embed_model,
            use_fp16=self.ngpu > 0,
            device=self.device,
        )
        self.rerankers = []
        for gpu_idx in range(max(1, self.ngpu)):
            device = f"cuda:{gpu_idx}" if self.ngpu else "cpu"
            self.rerankers.append(
                CrossEncoder(args.reranker_model, max_length=512, device=device)
            )
        print(
            f"Loaded {len(self.doc_texts):,} RAG documents; "
            f"nprobe={self.index.nprobe}; rerankers={len(self.rerankers)}"
        )

    def close(self):
        """Release every model reference before the retrieval process exits."""
        if hasattr(self, "embed_model"):
            del self.embed_model
        if hasattr(self, "rerankers"):
            self.rerankers.clear()
            del self.rerankers
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            try:
                torch.cuda.ipc_collect()
            except Exception:
                pass

    def _rerank_chunk(self, task):
        pairs, reranker_idx = task
        scores = self.rerankers[reranker_idx].predict(
            pairs,
            show_progress_bar=False,
            batch_size=self.args.rerank_batch_size,
        )
        return np.asarray(scores, dtype=np.float32).reshape(-1)

    def _parallel_rerank(self, pairs, ranges):
        device_count = max(1, len(self.rerankers))
        boundaries = np.linspace(0, len(ranges), device_count + 1, dtype=int)
        tasks = []
        for device_idx in range(device_count):
            row_start, row_end = boundaries[device_idx], boundaries[device_idx + 1]
            if row_start >= row_end:
                continue
            pair_start = ranges[row_start][0]
            pair_end = ranges[row_end - 1][1]
            tasks.append((pairs[pair_start:pair_end], device_idx))
        with ThreadPoolExecutor(max_workers=device_count) as executor:
            chunks = list(executor.map(self._rerank_chunk, tasks)) if tasks else []
        return np.concatenate(chunks) if chunks else np.zeros(0, dtype=np.float32)

    def retrieve_variants(self, queries):
        encoded = self.embed_model.encode(
            queries,
            return_dense=True,
            return_sparse=True,
            return_colbert_vecs=False,
            batch_size=self.args.encode_batch_size,
            max_length=512,
        )
        dense = np.asarray(encoded["dense_vecs"], dtype=np.float32)
        dense /= np.maximum(np.linalg.norm(dense, axis=1, keepdims=True), 1e-9)
        dense_scores, dense_indices = self.index.search(
            dense.astype(np.float32), self.args.top_k_retrieve
        )

        sparse_scores = np.zeros_like(dense_scores, dtype=np.float32)
        vocab_size = self.sparse_all.shape[1]
        for query_idx, lexical_weights in enumerate(encoded["lexical_weights"]):
            if not lexical_weights:
                continue
            columns = np.fromiter(lexical_weights.keys(), dtype=np.int64)
            values = np.fromiter(lexical_weights.values(), dtype=np.float32)
            query_sparse = sp.csr_matrix(
                (values, (np.zeros(len(columns), dtype=np.int32), columns)),
                shape=(1, vocab_size),
                dtype=np.float32,
            )
            scores = query_sparse.dot(self.sparse_all[dense_indices[query_idx]].T)
            sparse_scores[query_idx] = np.asarray(scores.toarray()).reshape(-1)

        sorted_indices = np.empty_like(dense_indices)
        for query_idx in range(len(queries)):
            hybrid = (
                self.args.alpha_dense * minmax(dense_scores[query_idx])
                + (1.0 - self.args.alpha_dense) * minmax(sparse_scores[query_idx])
            )
            sorted_indices[query_idx] = dense_indices[query_idx][np.argsort(-hybrid)]

        pairs = []
        ranges = []
        for query, indices in zip(queries, sorted_indices):
            start = len(pairs)
            pairs.extend([[query, self.doc_texts[idx]] for idx in indices])
            ranges.append((start, len(pairs)))
        all_scores = self._parallel_rerank(pairs, ranges)

        results = []
        for query_idx, indices in enumerate(sorted_indices):
            start, end = ranges[query_idx]
            ranked = sorted(
                zip(indices.tolist(), all_scores[start:end].tolist()),
                key=lambda item: -item[1],
            )[: self.args.variant_top_k]
            results.append([(self.doc_texts[idx], float(score)) for idx, score in ranked])
        return results

    def collect_candidates(self, specs):
        queries = []
        owners = []
        for spec_idx, spec in enumerate(specs):
            for query in spec["query_variants"]:
                queries.append(query)
                owners.append(spec_idx)
        print(f"Relation rows={len(specs)}; query variants={len(queries)}")
        variant_results = self.retrieve_variants(queries)
        candidate_sets = [dict() for _ in specs]
        for owner, results in zip(owners, variant_results):
            for document, score in results:
                candidate_sets[owner][document] = max(
                    score,
                    candidate_sets[owner].get(document, -1e9),
                )
        return candidate_sets

    def final_rerank(self, specs, candidate_sets):
        pairs = []
        ranges = []
        documents_by_spec = []
        for spec, candidate_map in zip(specs, candidate_sets):
            documents = list(candidate_map)
            documents_by_spec.append(documents)
            start = len(pairs)
            pairs.extend([[spec["canonical_query"], document] for document in documents])
            ranges.append((start, len(pairs)))
        scores = self._parallel_rerank(pairs, ranges)

        final_results = []
        for spec_idx, spec in enumerate(specs):
            start, end = ranges[spec_idx]
            ranked = []
            for document, score in zip(documents_by_spec[spec_idx], scores[start:end]):
                ranked.append(
                    {
                        "document": document,
                        "rerank_score": float(score),
                        **relation_flags(document, spec),
                    }
                )
            ranked.sort(
                key=lambda item: (
                    int(item["direct_relation_assertion"]),
                    int(item["target_relation_hit"]),
                    int(not item["cooccurrence_only"]),
                    item["rerank_score"],
                ),
                reverse=True,
            )
            final_results.append(ranked[: self.args.final_top_k])
        return final_results


def format_evidence(items):
    blocks = []
    for rank, item in enumerate(items, 1):
        if item.get("retrieval_mode") == "relation":
            header = (
                f'[Evidence {rank} | score={item["rerank_score"]:.4f} | '
                f'direct_pair={"yes" if item["direct_relation_assertion"] else "no"} | '
                f'cooccurrence_only={"yes" if item["cooccurrence_only"] else "no"}]'
            )
        else:
            header = f'[Evidence {rank} | score={item["rerank_score"]:.4f}]'
        blocks.append(f'{header} {item["document"]}')
    return "\n\n".join(blocks) if blocks else "[none]"


def filter_by_score(items, threshold):
    """Preserve reranker order while dropping evidence at or below threshold."""
    return [item for item in items if item["rerank_score"] > threshold]


def main():
    args = parse_args()
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    relation_df, general_df = load_classified_rag_rows(args.test_path, args.separated_dir)
    specs = []
    failed_ids = []
    for row in relation_df.itertuples(index=False):
        spec = make_query_spec(row)
        if spec is None:
            failed_ids.append(row.id)
        else:
            specs.append(spec)
    if failed_ids:
        raise ValueError(f"Could not build relation queries for IDs: {failed_ids[:20]}")
    if not specs and general_df.empty:
        raise ValueError("No classifier-selected rows require live RAG")

    started = time.time()
    retriever = RelationRetriever(args)
    try:
        if specs:
            candidate_sets = retriever.collect_candidates(specs)
            relation_ranked = retriever.final_rerank(specs, candidate_sets)
            for ranked in relation_ranked:
                for item in ranked:
                    item["retrieval_mode"] = "relation"
        else:
            candidate_sets = []
            relation_ranked = []

        if general_df.empty:
            general_ranked = []
        else:
            general_queries = general_df["prompt_bn"].apply(normalize_text).tolist()
            print("Ordinary classified RAG queries:", len(general_queries))
            general_raw = retriever.retrieve_variants(general_queries)
            general_ranked = [
                [
                    {
                        "document": document,
                        "rerank_score": float(score),
                        "retrieval_mode": "general",
                    }
                    for document, score in results[: args.final_top_k]
                ]
                for results in general_raw
            ]
    finally:
        retriever.close()
        del retriever
        gc.collect()
    print(f"All classified retrieval completed in {time.time() - started:.2f}s")

    top_ks = (2, 3, 4)
    relation_output_rows = {k: [] for k in top_ks}
    combined_output_rows = {k: [] for k in top_ks}
    relation_diagnostics = []
    combined_diagnostics = []
    for spec, candidate_map, ranked in zip(specs, candidate_sets, relation_ranked):
        common = {
            "id": spec["id"],
            "prompt_bn": spec["prompt_bn"],
            "response_bn": spec["response_bn"],
            "relation_type": spec["relation_type"],
            "target_word": spec["target_word"],
            "candidate_word": spec["candidate_word"],
            "retrieval_query": spec["canonical_query"],
            "retrieval_mode": "relation",
        }
        passing = filter_by_score(ranked, args.score_threshold)
        for top_k in top_ks:
            row = {**common, "context": format_evidence(passing[:top_k])}
            relation_output_rows[top_k].append(row)
            combined_output_rows[top_k].append(row)
        top3 = ranked[:3]
        top4 = ranked[:4]
        diagnostic = {
            **common,
            "classifier_source": relation_df.loc[
                relation_df["id"].map(id_key) == id_key(spec["id"]), "classifier_source"
            ].iloc[0],
            "query_variant_count": len(spec["query_variants"]),
            "unique_candidate_count": len(candidate_map),
            "candidate_in_top3": int(any(item["candidate_hit"] for item in top3)),
            "candidate_in_top4": int(any(item["candidate_hit"] for item in top4)),
            "direct_relation_in_top3": int(
                any(item["direct_relation_assertion"] for item in top3)
            ),
            "direct_relation_in_top4": int(
                any(item["direct_relation_assertion"] for item in top4)
            ),
            "cooccurrence_only_in_top3": int(
                any(item["cooccurrence_only"] for item in top3)
            ),
            "target_relation_in_top3": int(
                any(item["target_relation_hit"] for item in top3)
            ),
            "passing_count_0.75": len(passing),
            "top3_evidence": format_evidence(top3),
            "top4_evidence": format_evidence(top4),
        }
        relation_diagnostics.append(diagnostic)
        combined_diagnostics.append(diagnostic)

    for row, ranked in zip(general_df.itertuples(index=False), general_ranked):
        common = {
            "id": row.id,
            "prompt_bn": row.prompt_bn,
            "response_bn": row.response_bn,
            "relation_type": "",
            "target_word": "",
            "candidate_word": "",
            "retrieval_query": normalize_text(row.prompt_bn),
            "retrieval_mode": "general",
        }
        passing = filter_by_score(ranked, args.score_threshold)
        for top_k in top_ks:
            combined_output_rows[top_k].append(
                {**common, "context": format_evidence(passing[:top_k])}
            )
        combined_diagnostics.append(
            {
                **common,
                "classifier_source": row.classifier_source,
                "query_variant_count": 1,
                "unique_candidate_count": args.top_k_retrieve,
                "candidate_in_top3": "",
                "candidate_in_top4": "",
                "direct_relation_in_top3": "",
                "direct_relation_in_top4": "",
                "cooccurrence_only_in_top3": "",
                "target_relation_in_top3": "",
                "passing_count_0.75": len(passing),
                "top3_evidence": format_evidence(ranked[:3]),
                "top4_evidence": format_evidence(ranked[:4]),
            }
        )

    relation_frames = {
        top_k: pd.DataFrame(rows) for top_k, rows in relation_output_rows.items()
    }
    for top_k, frame in relation_frames.items():
        path = output_dir / f"lexical_relation_rag_top{top_k}_0.75.csv"
        frame.to_csv(path, index=False)
        hits = int((frame["context"] != "[none]").sum()) if not frame.empty else 0
        print(f"Wrote: {path} (> {args.score_threshold}; rows with evidence={hits}/{len(frame)})")

    relation_diagnostic = pd.DataFrame(relation_diagnostics)
    relation_diagnostic_path = output_dir / "lexical_relation_retrieval_diagnostic.csv"
    relation_diagnostic.to_csv(relation_diagnostic_path, index=False)
    print("Wrote:", relation_diagnostic_path)

    combined_frames = {
        top_k: pd.DataFrame(rows) for top_k, rows in combined_output_rows.items()
    }
    for top_k, frame in combined_frames.items():
        if not frame.empty and frame["id"].map(id_key).duplicated().any():
            raise ValueError(f"Combined top-{top_k} RAG contains duplicate IDs")
        path = output_dir / f"rag_output_top{top_k}_classified_0.75.csv"
        frame.to_csv(path, index=False)
        hits = int((frame["context"] != "[none]").sum()) if not frame.empty else 0
        print(f"Wrote: {path} (> {args.score_threshold}; rows with evidence={hits}/{len(frame)})")

    combined_diagnostic = pd.DataFrame(combined_diagnostics)
    combined_diagnostic_path = output_dir / "classified_rag_retrieval_diagnostic.csv"
    combined_diagnostic.to_csv(combined_diagnostic_path, index=False)
    print("Wrote:", combined_diagnostic_path)

    if not relation_diagnostic.empty:
        print("Specialized relation diagnostics:")
        for column in [
            "candidate_in_top3",
            "direct_relation_in_top3",
            "cooccurrence_only_in_top3",
            "target_relation_in_top3",
        ]:
            print(
                f"  {column}: "
                f"{int(relation_diagnostic[column].sum())}/{len(relation_diagnostic)}"
            )
    print("Live RAG rows written:", len(combined_frames[3]))


if __name__ == "__main__":
    main()


In [ ]:
# Run category-gated live retrieval only after deterministic categorization.
# The child process owns the BGE CUDA context. Its exit guarantees that the
# embedding model and rerankers retain zero GPU memory when vLLM starts.

import os
import subprocess
import sys
from pathlib import Path

relation_stage_script = Path("/kaggle/working/offline_relation_rag_stage.py")
relation_output_dir = Path(RELATION_RAG_OUTPUT_DIR)
relation_output_dir.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable,
    str(relation_stage_script),
    "--test-path", TEST_PATH,
    "--separated-dir", SEPARATED_DIR,
    "--index-dir", RAG_INDEX_DIR,
    "--embed-model", BGE_M3_PATH,
    "--reranker-model", RERANKER_PATH,
    "--output-dir", RELATION_RAG_OUTPUT_DIR,
    "--nprobe", "4096",
    "--top-k-retrieve", "30",
    "--variant-top-k", "10",
    "--final-top-k", "4",
    "--score-threshold", "0.75",
    "--alpha-dense", "0.7",
    "--encode-batch-size", "32",
    "--rerank-batch-size", "64",
]
rag_env = os.environ.copy()
rag_env.update({
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1",
    "TOKENIZERS_PARALLELISM": "false",
})
print("Running category-gated live RAG stage...")
subprocess.run(command, check=True, env=rag_env)

required_relation_outputs = [
    "lexical_relation_rag_top2_0.75.csv",
    "lexical_relation_rag_top3_0.75.csv",
    "lexical_relation_rag_top4_0.75.csv",
    "lexical_relation_retrieval_diagnostic.csv",
    "rag_output_top2_classified_0.75.csv",
    "rag_output_top3_classified_0.75.csv",
    "rag_output_top4_classified_0.75.csv",
    "classified_rag_retrieval_diagnostic.csv",
]
missing_relation_outputs = [
    name for name in required_relation_outputs
    if not (relation_output_dir / name).is_file()
]
if missing_relation_outputs:
    raise RuntimeError(f"Classified RAG stage did not produce: {missing_relation_outputs}")

# subprocess.run returned only after the retrieval process exited. Therefore its
# CUDA context, BGE-M3, and both rerankers have been destroyed by the OS.
print("Live RAG subprocess exited; embedding/reranker GPU memory released.")
print("Generated classified top-3 RAG used by Qwen:", RAG_CONTEXT_PATH)

# Informational only: there should be no retrieval Python process left here.
try:
    subprocess.run(
        ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv,noheader"],
        check=False,
        timeout=20,
    )
except Exception as exc:
    print("GPU process check skipped:", repr(exc))


In [ ]:
# Offline dependency installation

import importlib
import os
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


def install_vllm_offline(wheels_root):
    wheels_root = Path(wheels_root)
    if not wheels_root.exists():
        raise FileNotFoundError(f"Offline vLLM wheel directory not found: {wheels_root}")

    wheel_files = sorted(wheels_root.rglob("*.whl"))
    if not wheel_files:
        raise FileNotFoundError(f"No .whl files found under: {wheels_root}")

    wheel_dirs = sorted({str(path.parent) for path in wheel_files})
    command = [sys.executable, "-m", "pip", "install", "--no-index", "--upgrade"]
    for directory in wheel_dirs:
        command.extend(["--find-links", directory])
    command.append("vllm")

    print(f"Installing vLLM offline from {len(wheel_files)} wheel files...")
    subprocess.check_call(command)
    importlib.invalidate_caches()


if INSTALL_VLLM_FROM_WHEELS:
    install_vllm_offline(VLLM_WHEELS_DIR)

import vllm
print("vLLM version:", vllm.__version__)


In [ ]:
"""Offline four-bucket hallucination-detection inference pipeline."""

import gc
import importlib.util
import json
import os
import re
import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


In [ ]:
# Runtime controls

MODEL_ID = MODEL_PATH
TENSOR_PARALLEL_SIZE = int(os.environ.get("TENSOR_PARALLEL_SIZE", "2"))
GPU_MEMORY_UTILIZATION = float(os.environ.get("GPU_MEMORY_UTILIZATION", "0.95"))
MAX_MODEL_LEN = int(os.environ.get("MAX_MODEL_LEN", "6000"))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "96"))
MAX_GENERATION_TOKENS = int(os.environ.get("MAX_GENERATION_TOKENS", "780"))
VLLM_DTYPE = os.environ.get("VLLM_DTYPE", "half")
VLLM_QUANTIZATION = os.environ.get("VLLM_QUANTIZATION", "gptq" if "GPTQ" in MODEL_ID.upper() else "")
VLLM_ENFORCE_EAGER = os.environ.get("VLLM_ENFORCE_EAGER", "1") == "1"
VLLM_ENABLE_PREFIX_CACHING = os.environ.get("VLLM_ENABLE_PREFIX_CACHING", "1") == "1"
VLLM_ENABLE_CHUNKED_PREFILL = os.environ.get("VLLM_ENABLE_CHUNKED_PREFILL", "1") == "1"
VLLM_MAX_NUM_BATCHED_TOKENS = int(os.environ.get("VLLM_MAX_NUM_BATCHED_TOKENS", "8192"))
VLLM_MAX_NUM_SEQS = int(os.environ.get("VLLM_MAX_NUM_SEQS", str(max(BATCH_SIZE, 64))))
VLLM_SWAP_SPACE = int(os.environ.get("VLLM_SWAP_SPACE", "4"))
VLLM_KV_CACHE_DTYPE = os.environ.get("VLLM_KV_CACHE_DTYPE", "auto")
VLLM_ALL_PROMPTS_AT_ONCE = os.environ.get("VLLM_ALL_PROMPTS_AT_ONCE", "1") == "1"
MAX_RENDERED_PROMPT_TOKENS = int(
    os.environ.get(
        "MAX_RENDERED_PROMPT_TOKENS",
        str(max(1024, MAX_MODEL_LEN - max(MAX_GENERATION_TOKENS, 256) - 64)),
    )
)
SAFE_RENDERED_PROMPT_TOKENS = max(1024, MAX_MODEL_LEN - max(MAX_GENERATION_TOKENS, 256) - 64)
if MAX_RENDERED_PROMPT_TOKENS > SAFE_RENDERED_PROMPT_TOKENS:
    print(
        "Clipping MAX_RENDERED_PROMPT_TOKENS from",
        MAX_RENDERED_PROMPT_TOKENS,
        "to safe value",
        SAFE_RENDERED_PROMPT_TOKENS,
    )
    MAX_RENDERED_PROMPT_TOKENS = SAFE_RENDERED_PROMPT_TOKENS

ENABLE_THINKING = os.environ.get("ENABLE_THINKING", "0") == "1"
USE_LEXICAL_ARCHIVE = os.environ.get("USE_LEXICAL_ARCHIVE", "1") == "1"
USE_RAG_CONTEXT = os.environ.get("USE_RAG_CONTEXT", "1") == "1"
RAG_CONTEXT_DIR = str(Path(RAG_CONTEXT_PATH).parent)
RAG_CONTEXT_FILE = Path(RAG_CONTEXT_PATH).name
MAX_RAG_CONTEXT_CHARS = int(os.environ.get("MAX_RAG_CONTEXT_CHARS", "1200"))
LIMIT_ROWS = int(os.environ.get("LIMIT_ROWS", "0"))

MAX_CONTEXT_CHARS = int(os.environ.get("MAX_CONTEXT_CHARS", "5200"))

OUTPUT_DIR = Path(OUTPUT_DIR)
KAGGLE_ROOT = Path("/kaggle/input")
LOCAL_ROOT = Path(".")


print("Runtime config:")
for key, value in [
    ("MODEL_ID", MODEL_ID),
    ("TENSOR_PARALLEL_SIZE", TENSOR_PARALLEL_SIZE),
    ("GPU_MEMORY_UTILIZATION", GPU_MEMORY_UTILIZATION),
    ("MAX_MODEL_LEN", MAX_MODEL_LEN),
    ("BATCH_SIZE", BATCH_SIZE),
    ("MAX_GENERATION_TOKENS", MAX_GENERATION_TOKENS),
    ("VLLM_DTYPE", VLLM_DTYPE),
    ("VLLM_QUANTIZATION", VLLM_QUANTIZATION or "[auto]"),
    ("VLLM_ENFORCE_EAGER", VLLM_ENFORCE_EAGER),
    ("VLLM_ENABLE_PREFIX_CACHING", VLLM_ENABLE_PREFIX_CACHING),
    ("VLLM_ENABLE_CHUNKED_PREFILL", VLLM_ENABLE_CHUNKED_PREFILL),
    ("VLLM_MAX_NUM_BATCHED_TOKENS", VLLM_MAX_NUM_BATCHED_TOKENS),
    ("VLLM_MAX_NUM_SEQS", VLLM_MAX_NUM_SEQS),
    ("VLLM_SWAP_SPACE", VLLM_SWAP_SPACE),
    ("VLLM_KV_CACHE_DTYPE", VLLM_KV_CACHE_DTYPE),
    ("VLLM_ALL_PROMPTS_AT_ONCE", VLLM_ALL_PROMPTS_AT_ONCE),
    ("MAX_RENDERED_PROMPT_TOKENS", MAX_RENDERED_PROMPT_TOKENS),
    ("ENABLE_THINKING", ENABLE_THINKING),
    ("USE_LEXICAL_ARCHIVE", USE_LEXICAL_ARCHIVE),
    ("USE_RAG_CONTEXT", USE_RAG_CONTEXT),
    ("RAG_CONTEXT_DIR", RAG_CONTEXT_DIR),
    ("RAG_CONTEXT_FILE", RAG_CONTEXT_FILE),
    ("RAG_CONTEXT_PATH", RAG_CONTEXT_PATH),
    ("MAX_RAG_CONTEXT_CHARS", MAX_RAG_CONTEXT_CHARS),
    ("LEXICAL_ARCHIVE_PATH", LEXICAL_ARCHIVE_PATH or "[auto]"),
    ("LIMIT_ROWS", LIMIT_ROWS),
]:
    print(f"  {key}: {value}")


In [ ]:
# Data discovery helpers (used by separated pipeline)

def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    return None


def find_file_by_name(root, names):
    if not root.exists():
        return None
    lowered = {name.lower() for name in names}
    for path in root.rglob("*"):
        if path.is_file() and path.name.lower() in lowered:
            return path
    return None


In [ ]:
# Text helpers and categories

NO_CONTEXT_VALUES = {"", "nan", "NaN", "None", "none", "null", "NULL", "[NULL]", None}
BN_TO_EN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

LEXICAL_HINTS = ["শব্দার্থ", "ভাবার্থ", "বাগধারা", "বাগধারটির", "শাব্দিক অর্থ", "শব্দের অর্থ", "প্রবাদ"]
GRAMMAR_WORD_RELATION_HINTS = ["সমার্থক", "প্রতিশব্দ", "বিপরীতার্থক", "বিপরীত শব্দ"]
EXPLICIT_LEXICAL_HINTS = ["শাব্দিক অর্থ", "ভাবার্থ", "শব্দার্থ", "বাগধারা", "প্রবাদ"]
LEXICAL_REQUEST_PATTERNS = [
    r"শাব্দিক\s+অর্থ",
    r"ভাবার্থ",
    r"শব্দার্থ",
    r"বাগধার(?:া)?(?:টি|টির)?",
    r"প্রবাদ(?:টি|টির)?",
    r"(?:শব্দ(?:টি|টির)?|শব্দের)\s+(?:শুদ্ধ\s+)?(?:বাংলা\s+)?(?:অর্থ|মানে)",
    r"(?:কথা(?:টি|টির)?|কথার)\s+(?:বাংলা\s+)?অর্থ",
    r"[\"'][^\"']{1,100}[\"']\s*(?:[-–—]\s*)?(?:এর\s*)?(?:অর্থ|মানে|ভাবার্থ)",
    r"^.{1,80}?\s+এর\s+(?:শাব্দিক\s+)?(?:অর্থ|মানে|ভাবার্থ)\s*(?:কী|কি|বলতে)",
]
LEXICAL_NON_MEANING_HINTS = [
    "রচয়িতা",
    "রচয়িতা",
    "লেখক কে",
    "কবি কে",
    "উপসর্গটি কোন শ্রেণির",
    "কোন সমাস",
    "সন্ধিতে কোন শব্দ",
    "শুদ্ধ বানান",
]
LEXICAL_TERM_REJECT_HINTS = [
    "কত সালে",
    "কবে",
    "কে রচনা",
    "রচয়িতা",
    "রচয়িতা",
    "কোন সমাস",
    "উপসর্গটি",
    "সন্ধিতে",
    "নিচের কোনটি",
]
TRANSLATION_HINTS = ["অনুবাদ", "translate", "ইংরেজি", "english"]
MATH_HINTS = ["সম্ভাবনা", "সংখ্যা", "যোগ", "বিয়োগ", "বিয়োগ", "গুণ", "ভাগ", "ভগ্নাংশ", "শতকরা", "%", "/", "=", "কোণ", "ত্রিভুজ"]
MCQ_HINTS = ["ক)", "খ)", "গ)", "ঘ)", "(ক)", "(খ)", "(গ)", "(ঘ)", "নিচের কোন", "কোনটি"]
YEAR_HINTS = ["সাল", "সালে", "বছর", "কবে", "কত সালে"]


def clean_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def clean_context(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    return "" if text in NO_CONTEXT_VALUES else text


def normalize_space(text):
    return re.sub(r"\s+", " ", clean_text(text)).strip()


def normalize_key(text):
    text = normalize_space(text).translate(BN_TO_EN_DIGITS)
    return text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")


def compact(text, max_chars):
    text = normalize_space(text)
    if len(text) <= max_chars:
        return text
    head = text[: max_chars // 2]
    tail = text[-max_chars // 2 :]
    return head + " ... " + tail


def rag_path_candidates():
    return [Path(RAG_CONTEXT_PATH)]


def find_rag_context_path():
    path = Path(RAG_CONTEXT_PATH)
    return path if path.exists() and path.is_file() else None


def load_rag_context_map():
    if not USE_RAG_CONTEXT:
        print("RAG context disabled.")
        return {}
    path = find_rag_context_path()
    if path is None:
        tried = [str(path) for path in rag_path_candidates()]
        print("RAG context file not found; continuing without RAG context.")
        print("Tried RAG paths:", tried[:8], "..." if len(tried) > 8 else "")
        return {}
    print("RAG_CONTEXT_PATH:", path)
    rag_df = pd.read_csv(path)
    required = {"id", "context"}
    missing = required - set(rag_df.columns)
    if missing:
        print("RAG context missing required columns:", sorted(missing), "; continuing without RAG context.")
        return {}
    rag_df = rag_df[["id", "context"]].copy()
    rag_df["id_key"] = rag_df["id"].astype(str)
    rag_df["rag_context"] = rag_df["context"].apply(clean_context)
    rag_df = rag_df[rag_df["rag_context"].str.len() > 0]
    rag_map = dict(zip(rag_df["id_key"], rag_df["rag_context"]))
    print("Loaded RAG context rows:", len(rag_map))
    return rag_map


def attach_rag_context(df, rag_map):
    df = df.copy()
    if "rag_context" not in df.columns:
        df["rag_context"] = ""
    if not rag_map:
        return df
    df["rag_context"] = df["id"].astype(str).map(rag_map).fillna("").apply(clean_context)
    print("Rows with RAG context:", int((df["rag_context"].str.len() > 0).sum()), "of", len(df))
    return df


RETRIEVED_TAG_PATTERN = re.compile(r"(?<![A-Za-z])<(NH|H)>(?![A-Za-z])", re.IGNORECASE)
RAG_EVIDENCE_SPLIT_PATTERN = re.compile(
    r"(?=(?:\[Evidence\s+\d+\s*\||Retrieved Knowledge:\s*Score\s*\())",
    re.IGNORECASE,
)


def split_rag_evidence(raw_rag):
    raw_rag = clean_context(raw_rag)
    if not raw_rag:
        return []
    parts = [part.strip() for part in RAG_EVIDENCE_SPLIT_PATTERN.split(raw_rag) if part.strip()]
    return parts or [raw_rag]


def question_match_key(text):
    text = normalize_key(text).casefold()
    text = re.sub(r"[\W_]+", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()


def retrieved_tag_info_for_row(row):
    """Return the first ranked tag whose evidence contains this exact normalized question."""
    if not USE_RAG_CONTEXT:
        return "", ""
    raw_rag = clean_context(getattr(row, "rag_context", ""))
    question_key = question_match_key(getattr(row, "prompt_bn", ""))
    if not raw_rag or not question_key:
        return "", ""
    for evidence in split_rag_evidence(raw_rag):
        match = RETRIEVED_TAG_PATTERN.search(evidence)
        if match and question_key in question_match_key(evidence):
            return f"<{match.group(1).upper()}>", compact(evidence, MAX_RAG_CONTEXT_CHARS)
    return "", ""


def retrieved_tag_for_row(row):
    return retrieved_tag_info_for_row(row)[0]


def strip_tagged_rag_records(raw_rag):
    kept = [
        evidence for evidence in split_rag_evidence(raw_rag)
        if RETRIEVED_TAG_PATTERN.search(evidence) is None
    ]
    return "\n\n".join(kept)


def rag_context_for_row(row):
    if not USE_RAG_CONTEXT or row.category == "lexical_meaning":
        return "[none]"
    rag = clean_context(getattr(row, "rag_context", ""))
    if not rag:
        return "[none]"
    return compact(rag, MAX_RAG_CONTEXT_CHARS)


def has_any(text, needles):
    lower = normalize_key(text).lower()
    return any(n.lower() in lower for n in needles)


def has_direct_meaning_pattern(prompt):
    prompt = normalize_key(prompt)
    return any(re.search(pattern, prompt, flags=re.IGNORECASE) for pattern in LEXICAL_REQUEST_PATTERNS)


def is_grammar_word_relation_prompt(prompt):
    return has_any(prompt, GRAMMAR_WORD_RELATION_HINTS)


def is_lexical_prompt(prompt):
    prompt = normalize_key(prompt)
    if is_grammar_word_relation_prompt(prompt):
        return False
    if has_any(prompt, LEXICAL_NON_MEANING_HINTS) and not has_any(prompt, EXPLICIT_LEXICAL_HINTS):
        return False
    return has_direct_meaning_pattern(prompt)


def valid_lexical_term(term):
    term = normalize_space(term).strip(" ?।:-–—\"'")
    if not (1 < len(term) <= 100):
        return False
    if len(term.split()) > 12 or any(value in term for value in ["?", "|", "http://", "https://"]):
        return False
    return not has_any(term, LEXICAL_TERM_REJECT_HINTS)


def extract_lexical_term(prompt):
    prompt = normalize_key(prompt)
    if not is_lexical_prompt(prompt) and not is_grammar_word_relation_prompt(prompt):
        return ""

    quoted = re.search(r'[\"\']([^\"\']{1,100})[\"\']', prompt)
    if quoted:
        term = quoted.group(1).strip(" ?।:-–—\"'")
        return term if valid_lexical_term(term) else ""

    patterns = [
        r"^(?:এখানে\s+)?(.{1,80}?)\s+(?:শব্দ(?:টি|টির)?|শব্দের)\s+(?:শুদ্ধ\s+)?(?:বাংলা\s+)?(?:অর্থ|মানে|সমার্থক|প্রতিশব্দ|বিপরীতার্থক|বিপরীত\s+শব্দ)",
        r"^(?:এখানে\s+)?(.{1,80}?)[–—-]\s*(?:বাগধার(?:া)?|প্রবাদ)",
        r"^(?:এখানে\s+)?(.{1,80}?)\s+এর\s+(?:শাব্দিক\s+)?(?:অর্থ|মানে|ভাবার্থ)",
        r"^(?:এখানে\s+)?(.{1,80}?)\s+(?:বাগধার(?:া)?|প্রবাদ)(?:টি|টির)?\s+(?:অর্থ|মানে)",
        r"^(?:এখানে\s+)?(.{1,80}?)\s+(?:কথা(?:টি|টির)?|কথার)\s+(?:বাংলা\s+)?অর্থ",
    ]
    for pattern in patterns:
        match = re.search(pattern, prompt)
        if match:
            term = match.group(1).strip(" ?।:-–—\"'")
            if valid_lexical_term(term):
                return term
    return ""


def meaning_type(prompt):
    prompt = normalize_key(prompt)
    if "বিপরীতার্থক" in prompt or "বিপরীত শব্দ" in prompt:
        return "antonym"
    if "সমার্থক" in prompt or "প্রতিশব্দ" in prompt:
        return "synonym"
    if not is_lexical_prompt(prompt):
        return ""
    if "ভাবার্থ" in prompt or "বাগধারা" in prompt or "বাগধারটির" in prompt or "প্রবাদ" in prompt:
        return "figurative"
    if "শাব্দিক" in prompt:
        return "literal"
    if extract_lexical_term(prompt):
        return "literal_or_common"
    return ""


def categorize_row(context, prompt):
    if clean_context(context):
        return "context_grounded"
    if is_lexical_prompt(prompt) and extract_lexical_term(prompt):
        return "lexical_meaning"
    if has_any(prompt, TRANSLATION_HINTS):
        return "translation"
    if has_any(prompt, MATH_HINTS):
        return "math"
    if has_any(prompt, MCQ_HINTS):
        return "mcq"
    if has_any(prompt, YEAR_HINTS):
        return "year"
    return "general"


def is_negative_question(prompt):
    return has_any(prompt, ["নয়", "নয়", "ভুল", "বেমানান", "ব্যতিক্রম", "not", "wrong", "except"])


def requested_answer_type(prompt, category):
    prompt = normalize_key(prompt)
    if "বিপরীতার্থক" in prompt or "বিপরীত শব্দ" in prompt:
        return "grammar_antonym"
    if "সমার্থক" in prompt or "প্রতিশব্দ" in prompt:
        return "grammar_synonym"
    if category == "context_grounded":
        if has_any(prompt, YEAR_HINTS):
            return "context_year_or_date"
        if has_any(prompt, ["কত", "সংখ্যা", "কতটি", "কতগুলো", "কত জন", "কত কিলোমিটার"]):
            return "context_exact_number"
        if has_any(prompt, ["কে", "কার", "কোন", "নাম"]):
            return "context_name_or_entity"
        return "context_fact"
    if category == "math":
        return "math_exact_value"
    if category == "mcq":
        return "negative_mcq_or_exception" if is_negative_question(prompt) else "mcq_option"
    if category == "lexical_meaning":
        if "ভাবার্থ" in prompt or "বাগধারা" in prompt or "বাগধারটির" in prompt or "প্রবাদ" in prompt:
            return "lexical_figurative"
        if "শাব্দিক" in prompt or "শব্দার্থ" in prompt:
            return "lexical_literal"
        if "সমার্থক" in prompt:
            return "synonym"
        if "বিপরীতার্থক" in prompt:
            return "antonym"
        return "lexical_common_meaning"
    if has_any(prompt, YEAR_HINTS):
        return "year_or_date"
    if has_any(prompt, ["কে", "কার", "কোন লেখক", "কোন কবি", "রচয়িতা", "রচয়িতা", "সম্পাদনায়", "ছদ্মনাম"]):
        return "name_or_person"
    return category


def preprocess_frame(df, split):
    df = df.copy()
    for col in ["context", "prompt_bn", "response_bn"]:
        if col not in df.columns:
            df[col] = ""
    if "rag_context" not in df.columns:
        df["rag_context"] = ""
    if "id" not in df.columns:
        df["id"] = np.arange(1, len(df) + 1)
    df["split"] = split
    df["context"] = df["context"].apply(clean_context)
    df["rag_context"] = df["rag_context"].apply(clean_context)
    df["prompt_bn"] = df["prompt_bn"].apply(clean_text)
    df["response_bn"] = df["response_bn"].apply(clean_text)
    df["has_context"] = (df["context"].str.len() > 0).astype(int)
    df["prompt_key"] = df["prompt_bn"].apply(normalize_key)
    df["response_key"] = df["response_bn"].apply(normalize_key)
    df["prompt_response_key"] = df["prompt_key"] + " ||| " + df["response_key"]
    df["lexical_term"] = df["prompt_bn"].apply(extract_lexical_term)
    df["meaning_type"] = df["prompt_bn"].apply(meaning_type)
    df["category"] = [categorize_row(c, p) for c, p in zip(df["context"], df["prompt_bn"])]
    df["negative_question"] = df["prompt_bn"].apply(is_negative_question).astype(int)
    df["requested_answer_type"] = [
        requested_answer_type(p, cat) for p, cat in zip(df["prompt_bn"], df["category"])
    ]
    return df



In [ ]:
# Curated lexical / idiom archive. The archive is used as prompt evidence only,
# never as a hard label override.


def archive_key(text):
    text = normalize_key(text)
    text = text.replace("`", "'")
    text = re.sub(r"[\u200c\u200d]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip(" \t\r\n।,;:!?()[]{}\"'“”‘’`.-–—")


def archive_path_candidates():
    return [Path(LEXICAL_ARCHIVE_PATH)]


def iter_archive_json_files(path):
    path = Path(path)
    if path.is_file() and path.suffix.lower() == ".json":
        yield path
    elif path.is_dir():
        yield from sorted(path.glob("*.json"))


def merge_text_values(old_value, new_value):
    old_parts = [part.strip() for part in str(old_value or "").split(" | ") if part.strip()]
    new_parts = [part.strip() for part in str(new_value or "").split(" | ") if part.strip()]
    merged = []
    seen = set()
    for part in old_parts + new_parts:
        key = normalize_key(part)
        if key and key not in seen:
            seen.add(key)
            merged.append(part)
    return " | ".join(merged)


def merge_archive_entry(existing, new_entry):
    if existing is None:
        return new_entry
    merged = dict(existing)
    for field in [
        "archive_term",
        "archive_alternatives",
        "archive_literal_meaning",
        "archive_figurative_meaning",
        "archive_figurative_meaning_en",
        "archive_usage_domain",
        "archive_source_file",
    ]:
        merged[field] = merge_text_values(merged.get(field, ""), new_entry.get(field, ""))
    return merged


def load_lexical_archive():
    if not USE_LEXICAL_ARCHIVE:
        print("Lexical archive disabled.")
        return {}

    archive = {}
    loaded_files = 0
    indexed_terms = 0
    for root in archive_path_candidates():
        if not root.exists():
            continue
        for path in iter_archive_json_files(root):
            try:
                with open(path, encoding="utf-8") as f:
                    data = json.load(f)
            except Exception:
                continue
            if not isinstance(data, dict):
                continue

            main_term = clean_text(data.get("idiom", ""))
            alternatives = data.get("alternative_idioms", []) or []
            if isinstance(alternatives, str):
                alternatives = [alternatives]
            terms = [main_term] + [clean_text(x) for x in alternatives]
            entry = {
                "archive_term": main_term,
                "archive_alternatives": " | ".join([t for t in alternatives if t]),
                "archive_literal_meaning": clean_text(data.get("literal_meaning", "")),
                "archive_figurative_meaning": clean_text(data.get("figurative_meaning_bn", "")),
                "archive_figurative_meaning_en": clean_text(data.get("figurative_meaning_en", "")),
                "archive_usage_domain": " | ".join(data.get("usage_domain", []) or []),
                "archive_source_file": str(path.name),
            }
            loaded_files += 1
            for term in terms:
                key = archive_key(term)
                if not key:
                    continue
                archive[key] = merge_archive_entry(archive.get(key), entry)
                indexed_terms += 1
        if archive:
            # Prefer the first explicit path/dataset found instead of blending unrelated copies.
            break

    print("Lexical archive entries:", len(archive), "files:", loaded_files, "terms indexed:", indexed_terms)
    return archive


LEXICAL_ARCHIVE = load_lexical_archive()


def lexical_archive_entry_for_row(row):
    if not USE_LEXICAL_ARCHIVE or not row.lexical_term:
        return None
    keys = [
        archive_key(row.lexical_term),
        archive_key(row.prompt_bn.strip("\"'“”‘’")),
    ]
    for key in keys:
        if key in LEXICAL_ARCHIVE:
            return LEXICAL_ARCHIVE[key]
    return None


def lexical_archive_hint_for_row(row):
    entry = lexical_archive_entry_for_row(row)
    if not entry:
        return "[none]", None

    literal = compact(entry.get("archive_literal_meaning", ""), 260)
    figurative = compact(entry.get("archive_figurative_meaning", ""), 260)
    meaning = row.meaning_type or "literal_or_common"
    if meaning == "figurative":
        instruction = "Prompt asks ভাবার্থ/বাগধারা: judge by figurative/idiomatic meaning, not literal word-by-word meaning."
    elif meaning == "literal":
        instruction = "Prompt asks শাব্দিক অর্থ: judge by literal/common meaning only; figurative meaning alone is not enough."
    elif meaning == "synonym":
        instruction = "Prompt asks synonym: use archive meanings only to understand the term, then judge synonym correctness."
    elif meaning == "antonym":
        instruction = "Prompt asks antonym: use archive meanings only to understand the term, then judge opposite-meaning correctness."
    else:
        instruction = "Prompt asks শব্দার্থ/অর্থ/মানে: prefer literal/common meaning unless wording clearly asks idiomatic meaning."

    hint = f"""Lexical archive hint:
Term: {entry.get('archive_term', row.lexical_term)}
Literal/common meaning: {literal or '[missing]'}
Figurative/idiomatic meaning: {figurative or '[missing]'}
{instruction}"""
    return hint, entry


In [ ]:
# vLLM setup

from tqdm.auto import tqdm
from vllm import LLM, SamplingParams

llm = None
tokenizer = None


def load_vllm(model_id):
    kwargs = {
        "model": model_id,
        "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
        "gpu_memory_utilization": GPU_MEMORY_UTILIZATION,
        "trust_remote_code": True,
        "dtype": VLLM_DTYPE,
        "enforce_eager": VLLM_ENFORCE_EAGER,
        "max_model_len": MAX_MODEL_LEN,
        "max_num_seqs": VLLM_MAX_NUM_SEQS,
        "max_num_batched_tokens": VLLM_MAX_NUM_BATCHED_TOKENS,
        "swap_space": VLLM_SWAP_SPACE,
        "kv_cache_dtype": VLLM_KV_CACHE_DTYPE,
        "disable_log_stats": True,
    }
    if VLLM_QUANTIZATION:
        kwargs["quantization"] = VLLM_QUANTIZATION
    if VLLM_ENABLE_PREFIX_CACHING:
        kwargs["enable_prefix_caching"] = True
    if VLLM_ENABLE_CHUNKED_PREFILL:
        kwargs["enable_chunked_prefill"] = True
    try:
        model = LLM(**kwargs)
    except Exception as exc:
        optional_keys = [
            "enable_prefix_caching",
            "enable_chunked_prefill",
            "max_num_seqs",
            "max_num_batched_tokens",
            "swap_space",
            "kv_cache_dtype",
            "quantization",
        ]
        if not any(key in kwargs for key in optional_keys):
            raise
        print("Fast-path vLLM kwargs failed; retrying without optional speed knobs:", repr(exc))
        for key in optional_keys:
            kwargs.pop(key, None)
        model = LLM(**kwargs)
    return model, model.get_tokenizer()


def unload_vllm():
    global llm, tokenizer
    try:
        del llm
        del tokenizer
    except Exception:
        pass
    llm = None
    tokenizer = None
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception:
        pass


def apply_chat(messages, enable_thinking=False):
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def trim_rendered_prompt(prompt, max_tokens=MAX_RENDERED_PROMPT_TOKENS):
    if max_tokens <= 0:
        return prompt, 0, 0, False
    try:
        token_ids = tokenizer.encode(prompt, add_special_tokens=False)
    except TypeError:
        token_ids = tokenizer.encode(prompt)
    if len(token_ids) <= max_tokens:
        return prompt, len(token_ids), len(token_ids), False

    head_tokens = min(2200, max_tokens // 3)
    tail_tokens = max_tokens - head_tokens
    trimmed_ids = token_ids[:head_tokens] + token_ids[-tail_tokens:]
    trimmed = tokenizer.decode(trimmed_ids, skip_special_tokens=False)
    return trimmed, len(token_ids), len(trimmed_ids), True


def trim_rendered_prompts(prompts, desc):
    trimmed_prompts = []
    trim_count = 0
    max_before = 0
    max_after = 0
    for prompt in prompts:
        trimmed, before, after, did_trim = trim_rendered_prompt(prompt)
        trimmed_prompts.append(trimmed)
        trim_count += int(did_trim)
        max_before = max(max_before, before)
        max_after = max(max_after, after)
    if trim_count:
        print(
            f"{desc}: trimmed {trim_count}/{len(prompts)} rendered prompts "
            f"(max before={max_before}, max after={max_after}, limit={MAX_RENDERED_PROMPT_TOKENS})."
        )
    return trimmed_prompts


# Sampling, parsing, and generation (shared by separated pipeline)


def make_label_sampling_params():
    return SamplingParams(
        temperature=0.0,
        top_p=1.0,
        top_k=-1,
        max_tokens=MAX_GENERATION_TOKENS,
        repetition_penalty=1.2,
        n=1,
    )


sampling_params = make_label_sampling_params()
final_label_pattern = re.compile(r"FINAL_LABEL\s*=\s*\\boxed\s*\{\s*([01])\s*\}", re.IGNORECASE)
box_pattern = re.compile(r"\\boxed\s*\{\s*([01])\s*\}")
math_mismatch_pattern = re.compile(
    r"wrong|incorrect|not correct|does not match|doesn't match|mismatch|response (?:is )?wrong|"
    r"উত্তর\s+ভুল|ভুল\s+উত্তর|মেলে\s+না|মিলছে\s+না|সঠিক\s+নয়|সঠিক\s+নয়|লেবেল\s*০|label should be 0",
    re.IGNORECASE,
)
math_match_pattern = re.compile(
    r"correct|matches|response (?:is )?right|সঠিক|মিলে\s+যায়|মিলে\s+যায়|label should be 1",
    re.IGNORECASE,
)


def parse_one_label(text, category):
    final_matches = final_label_pattern.findall(text)
    if final_matches:
        return int(final_matches[-1]), "final_label_box", True, 0

    boxed_matches = box_pattern.findall(text)
    if boxed_matches:
        return int(boxed_matches[-1]), "boxed", True, 0

    mismatch = int(bool(math_mismatch_pattern.search(text)))
    match = int(bool(math_match_pattern.search(text)))
    if category == "math":
        if mismatch:
            return 0, "math_mismatch_cue", False, mismatch
        if match and not mismatch:
            return 1, "math_match_cue", False, mismatch
        explicit_math_correct = re.search(
            r"response.*(?:matches|correct|right)|final answer.*(?:matches|correct)|"
            r"উত্তরটি\s+সঠিক|response.*সঠিক|মিলে\s+যায়|মিলে\s+যায়|হুবহু\s+মিলে|ঠিক\s+মিলে",
            text,
            flags=re.IGNORECASE,
        )
        if explicit_math_correct and not mismatch:
            return 1, "math_explicit_match_no_box", False, mismatch
        return 0, "math_default_zero_no_box", False, mismatch

    explicit = re.findall(r"(?:label|answer|final)\s*(?:should be|is|=|:)?\s*([01])", text, flags=re.IGNORECASE)
    if explicit:
        return int(explicit[-1]), "explicit_label_text", False, mismatch

    nums = re.findall(r"(?<!\d)([01])(?!\d)", text)
    return (int(nums[-1]) if nums else 0), "digit_fallback", False, mismatch


def parse_boxed_labels(outputs, categories=None):
    labels = []
    raw_texts = []
    parser_sources = []
    has_boxed = []
    mismatch_cues = []
    if categories is None:
        categories = [""] * len(outputs)
    for out, category in zip(outputs, categories):
        text = out.outputs[0].text.strip()
        raw_texts.append(text)
        label, source, boxed, mismatch = parse_one_label(text, category)
        labels.append(label)
        parser_sources.append(source)
        has_boxed.append(int(boxed))
        mismatch_cues.append(int(mismatch))
    return np.asarray(labels, dtype=int), raw_texts, parser_sources, has_boxed, mismatch_cues


def generate_label_outputs(prompts, desc):
    prompts = trim_rendered_prompts(prompts, desc)
    if VLLM_ALL_PROMPTS_AT_ONCE:
        print(f"{desc}: handing {len(prompts)} prompts to native vLLM scheduler.")
        return llm.generate(prompts, sampling_params, use_tqdm=True)

    outputs = []
    for start in tqdm(range(0, len(prompts), BATCH_SIZE), desc=desc):
        batch_prompts = prompts[start : start + BATCH_SIZE]
        outputs.extend(llm.generate(batch_prompts, sampling_params))
    return outputs


# Separated-CSV pipeline (math / grammar / no-context GK / context-grounded GK)

Routes each row to one of four dedicated system prompts based on which pre-split CSV it came from, then merges and sorts the results by `id`.

Expected layout:
```
separated/
  math.csv
  only_bangla.csv
  null_only_gk.csv
  non_null_only_gk.csv
```

Each CSV needs columns: `id`, `context`, `prompt_bn`, `response_bn`. Run `run_separated_pipeline()` to execute.


In [ ]:
# Bucket-specific system prompts (one per separated CSV)

MATH_SYSTEM_PROMPT = """তুমি একজন বাংলা গাণিতিক hallucination detection classifier।
Prompt-এ একটি গাণিতিক সমস্যা আছে (শতকরা, লাভ-ক্ষতি, বয়স, অনুপাত, সুদ, গড়, সম্ভাবনা, কাজ-সময়, ঐকিক নিয়ম ইত্যাদি)। তোমার কাজ Response-এর চূড়ান্ত সংখ্যাগত উত্তর সঠিক কিনা যাচাই করা।

Label:
0 = ভুল হিসাব বা ভুল চূড়ান্ত উত্তর
1 = সঠিক চূড়ান্ত উত্তর

Rules:
    ***ভাগশেষ যাচাইকরণ — একলাইন নিয়ম:** ভাগশেষ বের করার পর অবশ্যই যাচাই করবে যে (ভাগফল × ভাজক + ভাগশেষ = মূল সংখ্যা) কিনা
    ****"সপ্তাহের দিন/বার-এর পার্থক্য বের করার সময় ভাগশেষ (R) বের করার পর একলাফে উত্তর না দিয়ে শুরুর দিন থেকে ১টি করে দিন গুনে (day-by-day) মোট R বার এগিয়ে চূড়ান্ত বার নির্ণয় করবে— প্রতিটি ধাপ আলাদাভাবে দেখাবে,**"
    **** যদি প্রশ্নে সঠিক উত্তর নির্ধারণের জন্য পর্যাপ্ত তথ্য না থাকে, তবে Response যাচাই করে দেখো; Response যদি প্রশ্নের তথ্যের ভিত্তিতে স্পষ্টভাবে ভুল, ভিত্তিহীন বা নিশ্চিতভাবে নির্ণয় করা যায় না এমন নির্দিষ্ট উত্তর দাবি করে তবে অবশ্যই লেবেল 0 দাও, অন্যথায় লেবেল 1 দাও।****

     সময় ও মিনিট গণনার ক্ষেত্রে সতর্কতা:
    "X মিনিট আগে সময় ছিল Y" বর্তনীর অর্থ → বর্তমান সময় = Y + X মিনিট (Y থেকে X বিয়োগ নয়)
    মিনিটকে ঘণ্টায় রূপান্তর: ৩০ মিনিট = ০.৫ ঘণ্টা (০.৩০ নয়), ৪৫ মিনিট = ০.৭৫ ঘণ্টা
    
    সুদ/মুনাফার হার বার্ষিক:
    "৩ বছরের মুনাফা-আসলে ৫৫০০ টাকা" → মোট মুনাফা বের করে সেটিকে বছর দিয়ে ভাগ করে বার্ষিক হার বের করতে হবে
    
    "টাকায় কয়টি" লাভ-ক্ষতি:
    "টাকায় ৩টি করে আম ক্রয়" = ১ টাকায় ৩টি আম, প্রতি পিসের দাম = ১/৩ টাকা
    "টাকায় ২টি করে বিক্রয়" = ১ টাকায় ২টি আম, প্রতি পিসের দাম = ১/২ টাকা
    
    শতকরা পরিবর্তন বনাম পূর্বের শতকরা মান:
    "ক্ষেত্রফলের শতকরা কত পরিবর্তন" = ((নতুন − পুরাতন) / পুরাতন) × ১০০ (পরিবর্তনের শতকরা)
    "ক্ষেত্রফল পূর্বের কত শতাংশ" = (নতুন / পুরাতন) × ১০০ (পূর্বের শতকরা)
    পার্থক্য বুঝে উত্তর যাচাই করো।
    
    গুণোত্তর ধারা ও প্যাটার্ন:
    গুণোত্তর ধারার n-তম পদ: n = ৮ হলে "৮ম পদ", n = ৯ হলে "৯ম পদ"
    প্যাটার্ন-ভিত্তিক সমস্যায় (যেমন 2×3=812) সম্পূর্ণ প্যাটার্ন বের করে তারপর উত্তর যাচাই করো
    বীজগণিতীয় সমীকরণ সমাধানের প্রতিটি ধাপ যাচাই করো (বিশেষ করে গুণ/ভাগের সময়)

ধাপ:
1. Response দেখার আগে নিজে ধাপে ধাপে হিসাব করো। প্রয়োজনীয় সূত্র ও প্রকৃত সংখ্যা বসিয়ে দেখাও, কোনো ধাপ এড়িয়ে যাবে না, মনে মনে হিসাব করবে না।
2. তোমার হিসাব করা চূড়ান্ত মানের সাথে Response-এর চূড়ান্ত মান তুলনা করো।
3. সাংখ্যিক মান হুবহু মিললে label 1, এক থেকে বেশি সিগনিফিক্যান্ট ডিজিটে বা যেকোনো মাত্রায় অমিল হলে label 0। ইউনিট বা ফরম্যাট আলাদা হওয়া সমস্যা না (যেমন "৩০ দিন" আর "৩০" একই মান)।
4. চূড়ান্ত মান ঠিক থাকলে মাঝের ধাপ অসম্পূর্ণ বা ভিন্ন পদ্ধতিতে দেখানো হলেও label 1, যদি না প্রশ্নে নির্দিষ্ট পদ্ধতি ব্যবহার করতে বলা হয়েছে।
5. প্যাটার্ন-ভিত্তিক সমস্যায় সম্পূর্ণ প্যাটার্ন বের করে তারপর উত্তর যাচাই করো।

উদাহরণ ১ (কাজ-সময়):
Prompt: রহিম একা একটি কাজ ৫৫ দিনে, করিম একা ৬৬ দিনে শেষ করতে পারে। দুজনে একসাথে কাজ করলে কত দিনে শেষ হবে?
Response: ৩০ দিন
বিশ্লেষণ: ১/৫৫ + ১/৬৬ = ৬/৩৩০ + ৫/৩৩০ = ১১/৩৩০ = ১/৩০, প্রত্যাশিত উত্তর ৩০ দিন। মিলে যায়।
FINAL_LABEL=\\boxed{1}

উদাহরণ 2 (সময় ও মিনিট — "X মিনিট আগে"):
Prompt: ৫০ মিনিট আগে সময় ছিল ৪টা বেজে ৪৫ মিনিট, ৬টা বাজতে আর কতক্ষণ সময় বাকি আছে?
Response: ২৫ মিনিট
বিশ্লেষণ: বর্তমান সময় = ৪:৪৫ + ৫০ মিনিট = ৫:৩৫। ৬:০০ − ৫:৩৫ = ২৫ মিনিট। Response-এর ২৫ মিনিট সঠিক।
FINAL_LABEL=\\boxed{1}

উদাহরণ 3 ("টাকায় কয়টি" লাভ-ক্ষতি):
Prompt: টাকায় তিনটি করে আম ক্রয় করে টাকায় ২টি করে বিক্রয় করলে শতকরা কত লাভ হবে?
Response: ৫০%
বিশ্লেষণ: ১ টাকায় ৩টি = প্রতি পিস ক্রয়মূল্য = ১/৩ টাকা। ১ টাকায় ২টি = প্রতি পিস বিক্রয়মূল্য = ১/২ টাকা। লাভ = ১/২ − ১/৩ = ১/৬ টাকা প্রতি পিসে। শতকরা লাভ = (১/৬) / (১/৩) × ১০০ = (১/৬ × ৩/১) × ১০০ = ০.৫ × ১০০ = ৫০%। Response-এর ৫০% সঠিক।
FINAL_LABEL=\\boxed{1}

উদাহরণ ৪ (একক রূপান্তর — বাংলা গণিত প্রচলন):
Prompt: সরল সুদের হার শতকরা কত টাকা হলে যে কোনো মূলধন ৮ বছরে সুদে-আসলে তিনগুণ হবে?
Response: ২৫ টাকা
বিশ্লেষণ: সরল সুদ = ৩P − P = ২P। সূত্র: ২P = (P × R × ৮)/১০০ → R = ২৫। বাংলা গণিত প্রচলনে "শতকরা কত টাকা" বলতে শতকরা হার বোঝায় এবং "২৫ টাকা" মানেই ২৫%। এটি সঠিক ও সমতুল্য প্রকাশ, শুধু একক ভিন্ন।
FINAL_LABEL=\\boxed{1}

উদাহরণ 5 (গুণোত্তর ধারা — n-তম পদ):
Prompt: 1/√2, 1, √2 …ধারাটির কোন পদ 8√2 হবে?
Response: ৯তম পদ
বিশ্লেষণ: a = 1/√2, r = 1 ÷ (1/√2) = √2। a_n = a·r^(n−1)। 8√2 = (1/√2)·(√2)^(n−1) → উভয়পক্ষে √2 গুণ: 8·2 = (√2)^(n−1)·√2 → 16 = (√2)^n → 2^4 = 2^(n/2) → n/2 = 4 → n = 8। n = ৮ হলেই এটি ৯ম পদ (কারণ n=১ হলে ১ম পদ)। Response-এর "৯তম পদ" সঠিক।
FINAL_LABEL=\\boxed{1}

Output rule:
২-৪ লাইনে হিসাব দেখাও, তারপর ঠিক একটি লাইনে FINAL_LABEL=\\boxed{0} অথবা FINAL_LABEL=\\boxed{1}। কখনো খালি বক্স দিবে না।
"""


GRAMMAR_SYSTEM_PROMPT = """তুমি একজন বাংলা ব্যাকরণ hallucination detection classifier।
Prompt-এ বাংলা ব্যাকরণ সংক্রান্ত প্রশ্ন আছে (সমাস, সন্ধি, বাগধারা/ভাবার্থ, প্রকৃতি-প্রত্যয়, কারক-বিভক্তি, ধ্বনি-বর্ণ, সমার্থক/বিপরীত শব্দ, বানান শুদ্ধি, ছন্দ-অলংকার ইত্যাদি)। এই প্রশ্নগুলোর সাধারণত একটিমাত্র টেক্সটবুক-স্বীকৃত সঠিক উত্তর থাকে।

Label:
0 = ব্যাকরণগত ভুল উত্তর
1 = সঠিক উত্তর

Special Rule:
**** শাব্দিক অর্থ বা ভাবার্থ সম্পর্কিত প্রশ্নের ক্ষেত্রে Response-এর অর্থ বা ব্যাখ্যা Context-এ প্রদত্ত উত্তরের সঙ্গে হুবহু মিলিয়ে যাচাই করবে***
**** বিপরীতার্থক শব্দ সম্পর্কিত প্রশ্নের ক্ষেত্রে যদি Response-এর শব্দটি Context/RAG-এ প্রদত্ত বিপরীতার্থক শব্দের সঙ্গে অর্থগতভাবে সমতুল্য (সমার্থক) হয় তবে লেবেল 1 দাও;অন্যথায় লেবেল 0 দাও***
ধাপ:
1. প্রশ্নের ঠিক কোন শব্দ/বাক্যাংশ নিয়ে জিজ্ঞাসা করা হয়েছে সেটা নির্দিষ্ট করে চিহ্নিত করো। ভিন্ন শব্দরূপ ভিন্ন উত্তর নিতে পারে, যেমন "হজযাত্রী" আর "হজযাত্রা" এক শব্দ নয়।
2. Context বা RAG-এ যদি ঠিক এই শব্দ/বাক্যাংশ নিয়েই নিয়ম দেওয়া থাকে, সেটাই প্রত্যাশিত উত্তর ধরো। Context/RAG যদি কাছাকাছি কিন্তু ভিন্ন শব্দরূপ নিয়ে কথা বলে তাহলে সেটা উপেক্ষা করে নিজের ব্যাকরণ জ্ঞান দিয়ে প্রত্যাশিত উত্তর নির্ধারণ করো।
3. Context/RAG না থাকলে বা অপ্রাসঙ্গিক হলে নিজের প্রমিত বাংলা ব্যাকরণ জ্ঞান ব্যবহার করো।
4. বাগধারার ভাবার্থ প্রশ্নে, Context-এ অর্থ দেওয়া থাকলে সেটাই মানদণ্ড, Response ভিন্ন বা আক্ষরিক অর্থ দিলে label 0।
5. পারিভাষিক টার্ম (যেমন "ষষ্ঠী তৎপুরুষ" বনাম "চতুর্থী তৎপুরুষ") না মিললে label 0, এমনকি Response-এর ব্যাসবাক্য বা ব্যাখ্যা আংশিক ঠিক থাকলেও।

উদাহরণ ১ (সমাস, ভিন্ন শব্দরূপে RAG থাকলে উপেক্ষা):
Prompt: 'হজযাত্রী' শব্দটি কোন সমাসের উদাহরণ?
RAG: শুধু 'হজযাত্রা'-র জন্য চতুর্থী তৎপুরুষ উল্লেখ আছে, 'হজযাত্রী' নিয়ে কিছু নেই
Response: হজের জন্য যাত্রী, ষষ্ঠী তৎপুরুষ
বিশ্লেষণ: RAG-এর তথ্য 'হজযাত্রা' নিয়ে, প্রশ্নের শব্দ 'হজযাত্রী' নয়, তাই irrelevant এবং উপেক্ষা করা হলো। নিজের জ্ঞানে 'হজের যাত্রী' ব্যাসবাক্য অনুযায়ী এটি ষষ্ঠী তৎপুরুষ, Response তাই বলছে।
FINAL_LABEL=\\boxed{1}

উদাহরণ ২ (বাগধারার ভাবার্থ):
Context: "ইলাজ" এর ভাবার্থ, বিনাক্ষতিতে কঠিন কার্যসিদ্ধি
Prompt: "ইলাজ" এর ভাবার্থ কী?
Response: চিকিৎসা বা প্রতিকার
বিশ্লেষণ: Context অনুযায়ী প্রত্যাশিত উত্তর "বিনাক্ষতিতে কঠিন কার্যসিদ্ধি", Response সম্পূর্ণ ভিন্ন আক্ষরিক অর্থ দিচ্ছে।
FINAL_LABEL=\\boxed{0}

Output rule:
২-৩ লাইনে reasoning, তারপর ঠিক একটি লাইনে FINAL_LABEL=\\boxed{0} অথবা FINAL_LABEL=\\boxed{1}।
"""


NO_CONTEXT_SYSTEM_PROMPT = """তুমি একজন বাংলা সাধারণ জ্ঞান hallucination detection classifier।
এই প্রশ্নে কোনো Context দেওয়া নেই ([NULL])। তোমাকে নিজের জ্ঞান, এবং RAG evidence দেওয়া থাকলে সেটাও ব্যবহার করে বিচার করতে হবে Response সঠিক কিনা।

Label:
0 = নিশ্চিতভাবে ভুল বা বানানো তথ্য
1 = সঠিক অথবা গ্রহণযোগ্য উত্তর

Strict Rule:
 ****Reasoning সর্বোচ্চ ২-৩টি সংক্ষিপ্ত বাক্য, একবার সিদ্ধান্তে পৌঁছালে তা পুনর্বিবেচনা বা 'তবে' দিয়ে বার বার উল্টানো যাবে না। Context থেকে দীর্ঘ উদ্ধৃতি repeat করা যাবে না ***

Special Rule:
 ***নামের সাথে যদি কোনো পদ বা উপাধি (যেমন Dr., Begum, জনাব, শ্রী, প্রফেসর,আল্লামা ইত্যাদি) যুক্ত/বাদ থাকে কিন্তু মূল নামটি একই থাকে, তাহলে সেই পদ/উপাধির উপস্থিতি বা অনুপস্থিতি Hallucination হিসেবে গণ্য হবে না— শুধুমাত্র মূল নামের সাথে মিল আছে কিনা তা যাচাই করবে।****
 ***যদি response  প্রদত্ত তথ্য এবং কনটেক্সটে প্রাপ্ত তথ্যের অর্থ একই হয় (হুবহু মিল না থাকলেও), তবে সেটিকে 'লেবেল ১' হিসেবে চিহ্নিত করুন। আর যদি তথ্যটি বিপরীত বা ভিন্ন অর্থ প্রকাশ করে, তবে সেটিকে 'লেবেল ০' হিসেবে চিহ্নিত করুন।"

গুরুত্বপূর্ণ নীতি, সততার সাথে মেনে চলো: তোমার নিজের জ্ঞান অসম্পূর্ণ বা ভুল হতে পারে, বিশেষত কম প্রচলিত/obscure তথ্যে।
1. RAG evidence প্রশ্নের exact entity-র সাথে সরাসরি সম্পর্কিত হলে, তোমার নিজের জ্ঞানের চেয়ে RAG evidence-কে বেশি priority দাও।
2. তুমি যদি নিজের জ্ঞানে সম্পূর্ণ নিশ্চিত না হও, বিশেষত disambiguation নাম, বিদেশি ভূগোল, নির্দিষ্ট তারিখ, কম পরিচিত ব্যক্তি/স্থান নিয়ে প্রশ্নে, এবং কোনো RAG evidence-ও না থাকে, তাহলে জোর করে ভুল বলে দিও না, label 1-এর দিকে bias করো।
3. একাধিক প্রচলিত variant বা সঠিক উত্তর থাকতে পারে এমন প্রশ্নে (সমার্থক শব্দ, MCQ-স্টাইল বিকল্প), Response তোমার প্রথম মনে আসা উত্তরের সাথে না মিললেও তা একটি valid বিকল্প হতে পারে কিনা বিবেচনা করো।
4. শুধুমাত্র তখনই label 0 দাও যখন Response স্পষ্টভাবে বানোয়াট, স্ববিরোধী, বা সুপরিচিত ও উচ্চ-নিশ্চয়তার তথ্যের সাথে সরাসরি সাংঘর্ষিক।
5. Response প্রত্যাশিত তথ্যের চেয়ে বেশি specific detail দিলে (যেমন শুধু সাল প্রত্যাশিত হলেও Response-এ দিন-মাস-সাল), মূল তথ্যের সাথে সাংঘর্ষিক না হলে শুধু এই কারণে label 0 দিও না।


উদাহরণ ১ (নিশ্চিত জ্ঞান):
Prompt: বাংলাদেশের বিজয় দিবস কত তারিখে?
Response: ১৬ ডিসেম্বর
বিশ্লেষণ: সুপরিচিত তথ্য, সরাসরি মেলে।
FINAL_LABEL=\\boxed{1}

উদাহরণ ২ (অনিশ্চিত/বিতর্কিত GK, RAG নেই):
Prompt: নিচের কোনটি গ্রীনহাউজ গ্যাস নয়?
Response: নাইট্রিক অক্সাইড (NO)
বিশ্লেষণ: এটি একাধিক বৈধ উত্তরযোগ্য একটি বিষয়, প্রশ্নকর্তার প্রত্যাশিত নির্দিষ্ট বিকল্প কী তা আমি নিশ্চিতভাবে জানি না এবং Response-এ স্পষ্ট contradiction নেই।
FINAL_LABEL=\\boxed{1}

উদাহরণ ৩ (স্পষ্ট ভুল, উচ্চ-নিশ্চয়তার তথ্যের সাথে সাংঘর্ষিক):
Prompt: বাংলাদেশের রাজধানী কী?
Response: চট্টগ্রাম
বিশ্লেষণ: সুপরিচিত ও নিশ্চিত তথ্যের সাথে সরাসরি সাংঘর্ষিক।
FINAL_LABEL=\\boxed{0}

উদাহরণ 4 (নামের সাথে পদ/উপাধি — Hallucination নয়):
Context: বেগম রোকেয়া সাখাওয়াত হোসেন ১৮৮০ সালে জন্মগ্রহণ করেন।
Retrieved Context: (empty)
Prompt: রোকেয়া সাখাওয়াত হোসেন কত সালে জন্মগ্রহণ করেন?
Response: বেগম রোকেয়া সাখাওয়াত হোসেন ১৮৮০ সালে জন্মগ্রহণ করেন।
বিশ্লেষণ: Prompt-এ "বেগম" উপাধি ছাড়া নাম উল্লেখ থাকলেও Response-এ "বেগম" যোগ করা হয়েছে— মূল নাম (রোকেয়া সাখাওয়াত হোসেন) ও সাল (১৮৮০) অভিন্ন, তাই এই উপাধি সংযোজন Hallucination নয়।
উত্তর: FINAL_LABEL=\\boxed{1}

Output rule:
২-৩ লাইনে reasoning লিখো, নিশ্চিত না হলে সেটা স্পষ্ট উল্লেখ করো, তারপর ঠিক একটি লাইনে FINAL_LABEL=\\boxed{0} অথবা FINAL_LABEL=\\boxed{1}।
"""


CONTEXT_GROUNDED_SYSTEM_PROMPT = """তুমি একজন বাংলা context-grounded hallucination detection classifier।
Prompt-এর সাথে একটি Context দেওয়া আছে (এবং সম্ভবত RAG evidence)। Response Context/RAG-এর তথ্যের সাথে সামঞ্জস্যপূর্ণ কিনা যাচাই করো।

Label:
0 = Context/RAG-এর তথ্যের সাথে Response সাংঘর্ষিক বা বানোয়াট তথ্য যুক্ত করেছে
1 = Response Context/RAG-এর তথ্যের সাথে সামঞ্জস্যপূর্ণ

Strict Rule:
 ****Reasoning সর্বোচ্চ ২-৩টি সংক্ষিপ্ত বাক্য, একবার সিদ্ধান্তে পৌঁছালে তা পুনর্বিবেচনা বা 'তবে' দিয়ে বার বার উল্টানো যাবে না। Context থেকে দীর্ঘ উদ্ধৃতি repeat করা যাবে না ***

Special Rule:
 ***যদি Context-এই প্রশ্নের উত্তর স্পষ্টভাবে উপস্থিত থাকে, তবে সিদ্ধান্ত নেওয়ার সময় Context-কেই সর্বোচ্চ অগ্রাধিকার দাও এবং RAG থেকে প্রাপ্ত তথ্য উপেক্ষা করো; Context-এর ভিত্তিতেই লেবেল নির্ধারণ করো।
 ***নামের সাথে যদি কোনো পদ বা উপাধি (যেমন Dr., Begum, জনাব, শ্রী, প্রফেসর,আল্লামা ইত্যাদি) যুক্ত/বাদ থাকে কিন্তু মূল নামটি একই থাকে, তাহলে সেই পদ/উপাধির উপস্থিতি বা অনুপস্থিতি Hallucination হিসেবে গণ্য হবে না— শুধুমাত্র মূল নামের সাথে মিল আছে কিনা তা যাচাই করবে।****
 ***যদি response  প্রদত্ত তথ্য এবং কনটেক্সটে প্রাপ্ত তথ্যের অর্থ একই হয় (হুবহু মিল না থাকলেও), তবে সেটিকে 'লেবেল ১' হিসেবে চিহ্নিত করুন। আর যদি তথ্যটি বিপরীত বা ভিন্ন অর্থ প্রকাশ করে, তবে সেটিকে 'লেবেল ০' হিসেবে চিহ্নিত করুন।"
ধাপ:
1. প্রথমে পুরো Context মনোযোগ দিয়ে পড়ো, Response দেখার আগে। Context লম্বা হলেও প্রশ্নের নির্দিষ্ট entity সংক্রান্ত সব বাক্য খুঁজে বের করো, উত্তর একটি বাক্যে সীমাবদ্ধ না থেকে পুরো Context জুড়ে ছড়ানো থাকতে পারে।
2. [Entity Match]: Prompt-এ নির্দিষ্ট নাম বা সত্তা উল্লেখ থাকলে, Context একই সত্তা নিয়ে কথা বলছে কিনা যাচাই করো। ভিন্ন সত্তা হলে Context সম্পূর্ণ উপেক্ষা করে নিজের জ্ঞান ব্যবহার করো।
3. Context প্রাসঙ্গিক ও উত্তরসম্পন্ন হলে শুধু Context-ই মানদণ্ড, বাইরের জ্ঞান ব্যবহার করবে না।
4. Context অপ্রাসঙ্গিক বা উত্তরহীন হলে নিজের সাধারণ জ্ঞান ব্যবহার করো, নিশ্চিত না হলে অহেতুক strict হবে না।
5. [সংখ্যা/সাল, কোনো soft matching নয়]: সাল, তারিখ, সংখ্যা, ID, যেকোনো ক্ষেত্রে Context/RAG-এর মান আর Response-এর মান আলাদা হলে, এক এককের পার্থক্যও (যেমন ৬০০ বনাম ৬০১), সরাসরি label 0। কাছাকাছি বলে ছাড় দিবে না।
6. Response মূল তথ্যের সাথে সাংঘর্ষিক নয় এমন অতিরিক্ত specific detail দিলে (Context-এ শুধু সাল থাকলেও Response সেই সালের মধ্যেই পড়া একটি নির্দিষ্ট তারিখ দিলে), শুধু এই সম্প্রসারণের কারণে label 0 দিও না। শুধুমাত্র প্রকৃত সাংঘর্ষিক বা যাচাইযোগ্যভাবে ভুল অংশের জন্য label 0 দাও। এই নিয়ম নিয়ম ৫-কে override করে না, নতুন সংখ্যা Context-এর সংখ্যার সাথে সাংঘর্ষিক হলে তা এখনও label 0।



উদাহরণ ১ (Context-এ উত্তর ছড়ানো, মিস করা যাবে না):
Context: গ্যালিলিও গ্যালিলি... আধুনিক পদার্থবিজ্ঞানের জনক... [দীর্ঘ প্যারাগ্রাফ]
Prompt: পদার্থবিজ্ঞানের জনক কে?
Response: গ্যালিলিও গ্যালিলি
বিশ্লেষণ: Context স্পষ্টভাবে "আধুনিক পদার্থবিজ্ঞানের জনক" বলছে গ্যালিলিওকে, Response তার সাথে মেলে। Context প্রাসঙ্গিক ও সরাসরি উত্তরসম্পন্ন হওয়ায় নিজের বাইরের জ্ঞান দিয়ে override করা হবে না।
FINAL_LABEL=\\boxed{1}

উদাহরণ ২ (স্ট্রিক্ট সংখ্যা মিল, কোনো ছাড় নেই):
Context: আলি ইবনে আবু তালিব (৬০০ – ৬৬১)
Prompt: আলী ইবনে আবি তালিব কবে জন্মগ্রহণ করেন?
Response: ৬০১ খ্রিস্টাব্দে
বিশ্লেষণ: Context অনুযায়ী প্রত্যাশিত উত্তর ৬০০, Response ৬০১ বলছে, এক এককের পার্থক্য হলেও সংখ্যাগত অমিল, ছাড় দেওয়া হবে না।
FINAL_LABEL=\\boxed{0}

উদাহরণ ৩ (অতিরিক্ত সাংঘর্ষিক-না-হওয়া detail গ্রহণযোগ্য):
Context: অভিষেক বন্দ্যোপাধ্যায় (জন্ম ১৯৮৭)
Prompt: অভিষেক বন্দ্যোপাধ্যায়ের জন্ম কবে?
Response: ৭ নভেম্বর, ১৯৮৭
বিশ্লেষণ: Context শুধু সাল ১৯৮৭ নিশ্চিত করে। Response সেই একই সালের মধ্যেই একটি নির্দিষ্ট তারিখ যোগ করেছে যা Context-এর সালের সাথে সাংঘর্ষিক নয়।
FINAL_LABEL=\\boxed{1}

উদাহরণ 4 (নামের সাথে পদ/উপাধি — Hallucination নয়):
Context: বেগম রোকেয়া সাখাওয়াত হোসেন ১৮৮০ সালে জন্মগ্রহণ করেন।
Retrieved Context: (empty)
Prompt: রোকেয়া সাখাওয়াত হোসেন কত সালে জন্মগ্রহণ করেন?
Response: বেগম রোকেয়া সাখাওয়াত হোসেন ১৮৮০ সালে জন্মগ্রহণ করেন।
বিশ্লেষণ: Prompt-এ "বেগম" উপাধি ছাড়া নাম উল্লেখ থাকলেও Response-এ "বেগম" যোগ করা হয়েছে— মূল নাম (রোকেয়া সাখাওয়াত হোসেন) ও সাল (১৮৮০) অভিন্ন, তাই এই উপাধি সংযোজন Hallucination নয়।
উত্তর: FINAL_LABEL=\\boxed{1}

Output rule:
২-3 লাইনে reasoning, তারপর ঠিক একটি লাইনে FINAL_LABEL=\\boxed{0} অথবা FINAL_LABEL=\\boxed{1}। খালি বক্স কখনো না।
"""


DEFAULT_SYSTEM_PROMPT = """তুমি বাংলা hallucination detector। Prompt-এর তুলনায় Response বিচার করো।

Label:
0 = ভুল, বিরোধী, বানানো, অপ্রাসঙ্গিক, বা চাওয়া উত্তর দেয়নি।
1 = সঠিক ও বিশ্বস্ত উত্তর।

প্রমাণের কঠোর অগ্রাধিকার:
1. Context প্রশ্নের exact entity ও চাওয়া তথ্যের সরাসরি উত্তর দিলে Context-ই সর্বোচ্চ ও authoritative প্রমাণ। তখন RAG বা নিজের জ্ঞান দিয়ে Context-supported Response reject করবে না।
2. Context [NULL], অপ্রাসঙ্গিক, অসম্পূর্ণ, বা exact উত্তর না দিলে কেবল relevant RAG ব্যবহার করো; wrong-entity বা দুর্বলভাবে সম্পর্কিত RAG উপেক্ষা করো।
3. Context উত্তর না দিলে এবং relevant RAG না থাকলে নিজের নিশ্চিত জ্ঞান ব্যবহার করো। নিশ্চিত না হলে কেবল অনুমানের ভিত্তিতে সঠিক Response reject করবে না।
4. Context ও RAG বিরোধ করলে Context-এ exact প্রশ্নের সরাসরি উত্তর থাকলে Context-কে অগ্রাধিকার দাও; Context উত্তর না দিলে বেশি relevant ও exact-entity RAG ব্যবহার করো।
- Response-কে label 0 দেবে কেবল valid evidence থাকলে এবং তুমি যথেষ্ট confident হলে: direct Context, relevant exact-entity RAG, যাচাইকৃত math, বা নিজের নিশ্চিত factual knowledge। Evidence দুর্বল/অপ্রাসঙ্গিক/পরস্পরবিরোধী হলে এবং Response ভুল বলে নিশ্চিত না হলে শুধু সন্দেহের ভিত্তিতে reject করবে না।
- Lexical evidence থাকলে প্রশ্নটি শাব্দিক অর্থ নাকি ভাবার্থ চায় তা ঠিকভাবে মিলাও।
- গণিত হলে RAG নয়, নিজে সংক্ষিপ্ত হিসাব করে Response-এর final value মিলাও।
- সমার্থক wording ও গ্রহণযোগ্য rounding ভুল নয়। জনসংখ্যা, উৎপাদন, ব্যবহারকারীর সংখ্যা ইত্যাদি সময়ভেদে পরিবর্তনশীল আনুমানিক পরিসংখ্যানে একই scale-এর ছোট ও বাস্তবসম্মত numeric পার্থক্য শুধু এই কারণে reject করবে না; যেমন ১ কোটি ১৮ লাখ বনাম ১ কোটি ২৫ লাখ।
- এই numeric tolerance গণিতের final value, নির্দিষ্ট সাল/তারিখ, আইনগত সীমা, পরীক্ষার score, বা Context-এ exact হিসেবে চাওয়া স্থির সংখ্যার ক্ষেত্রে প্রযোজ্য নয়।
- একই ব্যক্তিকে স্পষ্টভাবে বোঝালে পূর্ণ নামের বদলে পরিচিত সংক্ষিপ্ত নাম গ্রহণ করো; যেমন "কাজী নজরুল ইসলাম" বনাম "নজরুল"।
- Prompt কবে/তারিখ জিজ্ঞেস করলে Response-এ Context-এর সঠিক year বা date-এর সঠিক অংশ থাকলে শুধু পূর্ণ দিন-মাস-সাল না দেওয়ার কারণে reject করো না; যেমন "২০ মার্চ ১৯৭১" বনাম "১৯৭১ সালে" বা "২০ মার্চ"। তবে Prompt স্পষ্টভাবে পূর্ণ তারিখ চাইলে, বা Response-এ থাকা কোনো অংশ ভুল হলে label 0।
- শব্দার্থ/ভাবার্থে wording সামান্য ভিন্ন হলেও মূল অর্থ একই হলে গ্রহণ করো। কিন্তু শাব্দিক অর্থ ও ভাবার্থ একে অপরের বিকল্প নয়।

উদাহরণ ১ (সাধারণ জ্ঞান):
Context: [NULL] | Prompt: বাংলাদেশের বিজয় দিবস কত তারিখে? | Response: ১৬ ডিসেম্বর
বিশ্লেষণ: সুপরিচিত সঠিক উত্তরের সাথে Response মেলে।
উত্তর: \\boxed{1}

উদাহরণ ২ (relevant RAG):
Context: [NULL]
RAG: বকশীগঞ্জ উপজেলা ১৯৮৩ সালে প্রতিষ্ঠিত হয়।
Prompt: বকশীগঞ্জ উপজেলা কত সালে প্রতিষ্ঠিত হয়? | Response: ১৯৮৩
বিশ্লেষণ: RAG একই entity-র সরাসরি উত্তর দেয় এবং Response মেলে।
উত্তর: \\boxed{1}

উদাহরণ ৩ (সাল ভুল):
Context: বুনসং আরজতউইকুলের জন্ম ২ সেপ্টেম্বর ১৯৩৬।
Prompt: তিনি কবে জন্মগ্রহণ করেন? | Response: ২ সেপ্টেম্বর ১৯৬০
বিশ্লেষণ: Context-এর সাল ১৯৩৬, Response-এর ১৯৬০; contradiction।
উত্তর: \\boxed{0}

উদাহরণ ৪ (Context অসম্পূর্ণ):
Context: ২০২৫ সালে উপজেলা গঠনের প্রাথমিক প্রস্তাব গৃহীত হয়।
Prompt: উপজেলা কত সালে প্রতিষ্ঠিত হয়? | Response: ২০২৫
বিশ্লেষণ: প্রস্তাবের সাল প্রতিষ্ঠার সাল নয়; Context প্রশ্নের উত্তর দেয় না।
উত্তর: \\boxed{0}

উদাহরণ ৫ (বাগধারা):
Context: "ইলাজ"-এর ভাবার্থ বিনাক্ষতিতে কঠিন কার্যসিদ্ধি।
Prompt: "ইলাজ"-এর ভাবার্থ কী? | Response: চিকিৎসা বা প্রতিকার
বিশ্লেষণ: Response চাওয়া ভাবার্থের বদলে সাধারণ অর্থ দিয়েছে।
উত্তর: \\boxed{0}

উদাহরণ ৬ (গণিত, সঠিক):
Prompt: রহিম ৫৫ দিনে ও করিম ৬৬ দিনে কাজ শেষ করে। একসাথে কত দিন?
Response: ৩০ দিন
বিশ্লেষণ: ১/৫৫+১/৬৬=১১/৩৩০=১/৩০; সঠিক উত্তর ৩০ দিন।
উত্তর: \\boxed{1}

উদাহরণ ৭ (গণিত, ভুল):
Prompt: ৪০০ টাকার পণ্য ২৫% লাভে বিক্রয়মূল্য কত? | Response: ৪৫০ টাকা
বিশ্লেষণ: লাভ=৪০০×২৫%=১০০; মূল্য=৫০০, তাই Response ভুল।
উত্তর: \\boxed{0}

উদাহরণ ৮ (গণিত, ভগ্নাংশ):
Prompt: ক, খ, গ যথাক্রমে ৬, ৮, ২৪ দিনে কাজ শেষ করে। একসাথে কত দিন?
Response: ৩০ দিন
বিশ্লেষণ: ১/৬+১/৮+১/২৪=৮/২৪=১/৩; সময় ৩ দিন, তাই Response ভুল।
উত্তর: \\boxed{0}

উদাহরণ ৯ (সংক্ষিপ্ত নাম):
Context: বাংলাদেশের জাতীয় কবি কাজী নজরুল ইসলাম।
Prompt: জাতীয় কবি কে? | Response: নজরুল
বিশ্লেষণ: সংক্ষিপ্ত নামটি একই ব্যক্তিকে স্পষ্টভাবে বোঝায়।
উত্তর: \\boxed{1}

উদাহরণ ১০ (সঠিক year-only date):
Context: ঘটনাটি ২০ মার্চ ১৯৭১ সালে ঘটে।
Prompt: ঘটনাটি কবে ঘটে? | Response: ১৯৭১ সালে
বিশ্লেষণ: সঠিক সাল দেওয়া হয়েছে; পূর্ণ তারিখ না থাকলেও contradiction নয়।
উত্তর: \\boxed{1}

উদাহরণ ১১ (lexical paraphrase):
Context: "অকৃতদার" অর্থ অবিবাহিত পুরুষ।
Prompt: অর্থ কী? | Response: যে পুরুষ বিয়ে করেনি
বিশ্লেষণ: wording ভিন্ন হলেও মূল অর্থ একই।
উত্তর: \\boxed{1}

উদাহরণ ১২ (পরিবর্তনশীল আনুমানিক সংখ্যা):
Context: জেলার জনসংখ্যা আনুমানিক ১ কোটি ১৮ লাখ। | Prompt: জেলার বর্তমান জনসংখ্যা কত? | Response: প্রায় ১ কোটি ২৫ লাখ
বিশ্লেষণ: জনসংখ্যা পরিবর্তনশীল এবং উভয় সংখ্যা একই scale-এর কাছাকাছি estimate; শুধু ছোট পার্থক্যের জন্য reject নয়।
উত্তর: \\boxed{1}

উদাহরণ ১৩ (Context-এর অগ্রাধিকার):
Context: ২০১৬ টি২০ বিশ্বকাপের স্বাগতিক ছিল ভারত। | RAG: টুর্নামেন্টে শ্রীলঙ্কা অংশ নেয়। | Prompt: কোথায় অনুষ্ঠিত হয়? | Response: ভারত
বিশ্লেষণ: Context exact answer দেয়; RAG distractor।
উত্তর: \\boxed{1}"""

RAG_ADDENDUM = """

[RAG EVIDENCE হ্যান্ডলিং]
User content-এ "Retrieved Knowledge (RAG)" ব্লক থাকতে পারে, এটি স্বয়ংক্রিয় retrieval, score থাকলেও তা relevance-এর গ্যারান্টি না।

নিয়ম:
- RAG entry ব্যবহারের আগে যাচাই করো এটি প্রশ্নের ঠিক সেই entity/শব্দ/বিষয় নিয়ে কিনা। নাম, সংখ্যা, শব্দরূপ হুবহু মিলতে হবে, কাছাকাছি বা similar বিষয় যথেষ্ট না।
- প্রাসঙ্গিক ও সরাসরি উত্তরসম্পন্ন RAG entry পেলে Context-এর মতোই strict সংখ্যা/শব্দ মিল যাচাই করো।
- একাধিক উৎস (Context, একাধিক RAG entry) পরস্পরবিরোধী সংখ্যা/তথ্য দিলে এই priority অনুসারে প্রত্যাশিত উত্তর ঠিক করো: Context > সঠিক-entity ও সর্বোচ্চ score/সবচেয়ে বেশি repeat হওয়া RAG entry > নিজের জ্ঞান। Response-এর মান এর মধ্যে কোনো উৎসের সাথেই না মিললে label 0।
- ভুল entity, দুর্বল বা আংশিক lexical মিল, বা vague/irrelevant RAG entry সম্পূর্ণ উপেক্ষা করো।
- RAG না থাকলে বা [none] হলে কোনো signal হিসেবে ধরবে না।
"""

# RAG_ADDENDUM is appended only for the no_context and context_grounded buckets,
# matching prompt.md. Math and grammar already fold RAG/Context handling into
# their own instructions above, so they stay as-is.
BUCKET_RAG_ADDENDUM = {
    "math": "",
    "grammar": "",
    "no_context": RAG_ADDENDUM,
    "context_grounded": RAG_ADDENDUM,
    "default": "",
}

BUCKET_SYSTEM_PROMPTS = {
    "math": MATH_SYSTEM_PROMPT,
    "grammar": GRAMMAR_SYSTEM_PROMPT,
    "no_context": NO_CONTEXT_SYSTEM_PROMPT,
    "context_grounded": CONTEXT_GROUNDED_SYSTEM_PROMPT,
    "default": DEFAULT_SYSTEM_PROMPT,
}


In [ ]:
# Four classifier outputs

SEPARATED_BUCKET_FILES = {
    "math": "math.csv",
    "grammar": "only_bangla.csv",
    "no_context": "null_only_gk.csv",
    "context_grounded": "non_null_only_gk.csv",
}

SEPARATED_RAG_MAP = None


def find_separated_dir():
    path = Path(SEPARATED_DIR)
    return path if path.exists() and path.is_dir() else None


def find_separated_bucket_file(separated_dir, filename):
    direct = separated_dir / filename
    if direct.exists():
        return direct
    matches = sorted(separated_dir.rglob(filename))
    return matches[0] if matches else None


def load_separated_bucket(separated_dir, bucket_key, filename):
    path = find_separated_bucket_file(separated_dir, filename)
    if path is None:
        raise FileNotFoundError(f"[{bucket_key}] Missing classifier output: {filename} under {separated_dir}")

    df = pd.read_csv(path)
    if "id" not in df.columns:
        raise ValueError(f"[{bucket_key}] {path} has no id column")
    df["id"] = pd.to_numeric(df["id"], errors="raise").astype("Int64")
    if LIMIT_ROWS > 0:
        df = df.head(LIMIT_ROWS).copy()
    df = preprocess_frame(df, f"separated_{bucket_key}")
    df = attach_rag_context(df, SEPARATED_RAG_MAP or {})
    print(f"[{bucket_key}] loaded {len(df)} rows from {path}")
    return df


In [ ]:
# Bucket-specific user content and prediction

BUCKET_CATEGORY_MAP = {
    "math": "math",
    "grammar": "grammar",
    "no_context": "general",
    "context_grounded": "context_grounded",
    "default": "general",
}


TAG_RULE_ADDENDUM = r"""

[RETRIEVED TAG RULE]
এই rule কেবল আলাদা Retrieved Tag block থাকলে প্রযোজ্য। Block-টি code দ্বারা current question-এর সঙ্গে যাচাই করা।
Context, সাধারণ RAG ও lexical evidence prompt-এ দেখা গেলেও সেগুলো ব্যবহার বা বিশ্লেষণ করবে না।
<H> হলে সরাসরি FINAL_LABEL=\boxed{0}; <NH> হলে সরাসরি FINAL_LABEL=\boxed{1}; কোনো reasoning নয়।
"""


def build_user_content_bucket(row, bucket_key):
    retrieved_tag, tagged_record = retrieved_tag_info_for_row(row)
    tag_block = ""
    if retrieved_tag:
        tag_block = fr"""
Retrieved Tagged Record:
{tagged_record}

Retrieved Tag:
{retrieved_tag}

Verified tag present: visible Context/RAG/lexical evidence ignore করো; reasoning ছাড়া tag অনুযায়ী final label দাও।
"""

    rag_evidence = rag_context_for_row(row)
    context = row.context if int(row.has_context) == 1 else "[NULL]"
    archive_hint, _ = lexical_archive_hint_for_row(row)
    if bucket_key == "default":
        raw_rag = clean_context(getattr(row, "rag_context", ""))
        rag_evidence = compact(raw_rag, MAX_RAG_CONTEXT_CHARS) if raw_rag else "[none]"

    if bucket_key == "math":
        return fr"""Judge input:

Category:
{BUCKET_CATEGORY_MAP[bucket_key]}

Requested answer type:
{row.requested_answer_type}

Negative/exception question:
{int(row.negative_question)}
{tag_block}
Prompt:
{row.prompt_bn}

Response:
{row.response_bn}

Think carefully only when needed. For obvious supported/unsupported answers, do not write reasoning.
For math/no-context hard facts, calculate or recall the needed fact before deciding.
If reasoning is needed, write at most 2-4 short lines.
Never output an empty box like \boxed{{}}.
End with one filled final label: FINAL_LABEL=\boxed{{0}} or FINAL_LABEL=\boxed{{1}}.
Final answer:"""

    lexical_block = ""
    if row.lexical_term:
        lexical_block = f"""
Target term:
{row.lexical_term}

Requested meaning type:
{row.meaning_type or 'common'}
"""


    if bucket_key == "default":
        return fr"""{tag_block}
Context:
{compact(context, MAX_CONTEXT_CHARS)}

Retrieved Knowledge (RAG):
{rag_evidence}

Lexical evidence:
{archive_hint}

Prompt:
{row.prompt_bn}

Response:
{row.response_bn}

প্রয়োজনে সর্বোচ্চ ২-৩টি ছোট reasoning line লিখে বিচার করো। স্পষ্ট উত্তর হলে reasoning প্রয়োজন নেই।
শেষ লাইনে অবশ্যই FINAL_LABEL=\boxed{{0}} অথবা FINAL_LABEL=\boxed{{1}} লিখবে।"""

    if bucket_key == "grammar":
        return fr"""Judge input:

Category:
{BUCKET_CATEGORY_MAP[bucket_key]}

Requested answer type:
{row.requested_answer_type}

Negative/exception question:
{int(row.negative_question)}
{tag_block}
{lexical_block}
Lexical archive evidence:
{archive_hint}

Context:
{compact(context, MAX_CONTEXT_CHARS)}

Retrieved Knowledge (RAG):
{rag_evidence}

Prompt:
{row.prompt_bn}

Response:
{row.response_bn}

Context note: if Context directly answers or contradicts, use it. If Context is clearly irrelevant/incomplete, do not reject solely for lack of support; judge the exact prompt like the training samples.
No-context Grammar/GK/literature/history note: do not reject a concise textbook-style answer just because another variant exists. Mark 0 only when you are sure it contradicts the expected answer and you have clear knowledge.
Think carefully only when needed. For obvious supported/unsupported answers, do not write reasoning.
For math/no-context hard facts, calculate or recall the needed fact before deciding.
If reasoning is needed, write at most 2-4 short lines.
Never output an empty box like \boxed{{}}.
End with one filled final label: FINAL_LABEL=\boxed{{0}} or FINAL_LABEL=\boxed{{1}}.
Final answer:"""

    if bucket_key == "no_context":
        return fr"""Judge input:

Category:
{BUCKET_CATEGORY_MAP[bucket_key]}

Requested answer type:
{row.requested_answer_type}

Negative/exception question:
{int(row.negative_question)}
{tag_block}

Retrieved Knowledge (RAG):
{rag_evidence}

Prompt:
{row.prompt_bn}

Response:
{row.response_bn}

Context note: if Context directly answers or contradicts, use it. If Context is clearly irrelevant/incomplete, do not reject solely for lack of support; judge the exact prompt like the training samples.
No-context Grammar/GK/literature/history note: do not reject a concise textbook-style answer just because another variant exists. Mark 0 only when you are sure it contradicts the expected answer and you have clear knowledge.
Think carefully only when needed. For obvious supported/unsupported answers, do not write reasoning.
For math/no-context hard facts, calculate or recall the needed fact before deciding.
If reasoning is needed, write at most 2-4 short lines.
Never output an empty box like \boxed{{}}.
End with one filled final label: FINAL_LABEL=\boxed{{0}} or FINAL_LABEL=\boxed{{1}}.
Final answer:"""

    # context_grounded
    return fr"""Judge input:

Category:
{BUCKET_CATEGORY_MAP[bucket_key]}

Requested answer type:
{row.requested_answer_type}

Negative/exception question:
{int(row.negative_question)}
{tag_block}

Context:
{compact(context, MAX_CONTEXT_CHARS)}

Retrieved Knowledge (RAG):
{rag_evidence}

Prompt:
{row.prompt_bn}

Response:
{row.response_bn}

Context note: if Context directly answers or contradicts, use it. If Context is clearly irrelevant/incomplete, do not reject solely for lack of support; judge the exact prompt like the training samples.
No-context Grammar/GK/literature/history note: do not reject a concise textbook-style answer just because another variant exists. Mark 0 only when you are sure it contradicts the expected answer and you have clear knowledge.
Think carefully only when needed. For obvious supported/unsupported answers, do not write reasoning.
For math/no-context hard facts, calculate or recall the needed fact before deciding.
If reasoning is needed, write at most 2-4 short lines.
Never output an empty box like \boxed{{}}.
End with one filled final label: FINAL_LABEL=\boxed{{0}} or FINAL_LABEL=\boxed{{1}}.
Final answer:"""


def predict_bucket(df, bucket_key):
    system_content = BUCKET_SYSTEM_PROMPTS[bucket_key] + BUCKET_RAG_ADDENDUM[bucket_key]

    prompts = []
    retrieved_tags = []
    for _, row in df.iterrows():
        retrieved_tag = retrieved_tag_for_row(row)
        retrieved_tags.append(retrieved_tag)
        row_system_content = system_content + TAG_RULE_ADDENDUM if retrieved_tag else system_content
        messages = [
            {"role": "system", "content": row_system_content},
            {"role": "user", "content": build_user_content_bucket(row, bucket_key)},
        ]
        prompts.append(apply_chat(messages, enable_thinking=False if retrieved_tag else ENABLE_THINKING))

    print(f"Retrieved tag prompts [{bucket_key}]:", sum(bool(tag) for tag in retrieved_tags))

    outputs = generate_label_outputs(prompts, f"Predict [{bucket_key}]")
    parse_categories = ["math" if bucket_key == "math" else ""] * len(df)
    main_labels, _, _, _, _ = parse_boxed_labels(
        outputs, parse_categories
    )

    result = pd.DataFrame({
        "id": df["id"].to_numpy(),
        "label": np.asarray(main_labels, dtype=int),
    })
    del outputs, prompts, retrieved_tags
    gc.collect()
    return result


In [ ]:
# Run all four buckets, validate against the official test file, and write submission.csv


def load_official_test():
    test_df = pd.read_csv(TEST_PATH)
    required = {"id", "context", "prompt_bn", "response_bn"}
    missing = required - set(test_df.columns)
    if missing:
        raise ValueError(f"Official test file is missing columns: {sorted(missing)}")
    test_df = test_df.copy()
    test_df["id"] = pd.to_numeric(test_df["id"], errors="raise").astype("Int64")
    if test_df["id"].isna().any() or not test_df["id"].is_unique:
        raise ValueError("Official test IDs must be non-null and unique")
    test_df["_test_order"] = np.arange(len(test_df))
    print("Official test rows:", len(test_df))
    return test_df




def move_word_relation_rows_to_grammar(bucket_dfs):
    bucket_dfs = dict(bucket_dfs)
    moved = []
    for bucket_key in list(bucket_dfs):
        if bucket_key == "grammar":
            continue
        frame = bucket_dfs[bucket_key]
        mask = frame["prompt_bn"].apply(is_grammar_word_relation_prompt)
        if mask.any():
            moved.append(frame.loc[mask].copy())
            bucket_dfs[bucket_key] = frame.loc[~mask].copy()

    if moved:
        bucket_dfs["grammar"] = pd.concat(
            [bucket_dfs["grammar"], *moved], ignore_index=True
        )
    print("Synonym/antonym rows moved to grammar:", sum(len(df) for df in moved))
    return bucket_dfs

def add_default_fallback_bucket(official_test, bucket_dfs, rag_map):
    classified = pd.concat(
        [df[["id"]] for df in bucket_dfs.values()],
        ignore_index=True,
    )
    duplicate_ids = classified.loc[classified["id"].duplicated(keep=False), "id"].unique()
    if len(duplicate_ids):
        raise ValueError(
            "Classifier assigned IDs to multiple buckets: "
            f"{list(duplicate_ids[:10])}"
        )

    classified_ids = set(classified["id"].astype(int))
    fallback = official_test.loc[
        ~official_test["id"].astype(int).isin(classified_ids)
    ].drop(columns="_test_order").copy()

    if fallback.empty:
        print("Default fallback rows: 0")
        return bucket_dfs

    fallback = preprocess_frame(fallback, "separated_default")
    fallback = attach_rag_context(fallback, rag_map)
    bucket_dfs = dict(bucket_dfs)

    relation_mask = fallback["prompt_bn"].apply(is_grammar_word_relation_prompt)
    relation_rows = fallback.loc[relation_mask].copy()
    fallback = fallback.loc[~relation_mask].copy()
    if not relation_rows.empty:
        bucket_dfs["grammar"] = pd.concat(
            [bucket_dfs["grammar"], relation_rows], ignore_index=True
        )
    print("Fallback synonym/antonym rows routed to grammar:", len(relation_rows))

    if not fallback.empty:
        bucket_dfs["default"] = fallback
    print("Default fallback rows:", len(fallback))
    return bucket_dfs

def validate_classifier_partition(official_test, bucket_dfs):
    classified = pd.concat(
        [df[["id"]].assign(bucket=bucket) for bucket, df in bucket_dfs.items()],
        ignore_index=True,
    )
    duplicate_ids = classified.loc[classified["id"].duplicated(keep=False), "id"].unique()
    official_ids = set(official_test["id"].astype(int))
    classified_ids = set(classified["id"].astype(int))
    missing_ids = sorted(official_ids - classified_ids)
    extra_ids = sorted(classified_ids - official_ids)

    if len(duplicate_ids) or missing_ids or extra_ids or len(classified) != len(official_test):
        raise ValueError(
            "Classifier partition does not match the official test set. "
            f"duplicates={list(duplicate_ids[:10])}, "
            f"missing={missing_ids[:10]}, extra={extra_ids[:10]}, "
            f"classified_rows={len(classified)}, test_rows={len(official_test)}"
        )
    print("Classifier partition validation: PASS")


def run_separated_pipeline():
    global llm, tokenizer, SEPARATED_RAG_MAP

    official_test = load_official_test()
    separated_dir = find_separated_dir()
    if separated_dir is None:
        raise FileNotFoundError(f"Classifier output directory not found: {SEPARATED_DIR}")
    print("SEPARATED_DIR:", separated_dir)

    SEPARATED_RAG_MAP = load_rag_context_map()

    bucket_dfs = {}
    for bucket_key, filename in SEPARATED_BUCKET_FILES.items():
        bucket_dfs[bucket_key] = load_separated_bucket(separated_dir, bucket_key, filename)

    bucket_dfs = move_word_relation_rows_to_grammar(bucket_dfs)
    bucket_dfs = add_default_fallback_bucket(official_test, bucket_dfs, SEPARATED_RAG_MAP)
    validate_classifier_partition(official_test, bucket_dfs)

    print("=" * 80)
    print("Loading local Qwen model:", MODEL_ID)
    llm, tokenizer = load_vllm(MODEL_ID)
    try:
        bucket_predictions = [
            predict_bucket(df, bucket_key)
            for bucket_key, df in bucket_dfs.items()
        ]
    finally:
        print("Inference complete")

    merged = pd.concat(bucket_predictions, ignore_index=True)
    if merged["id"].isna().any() or not merged["id"].is_unique:
        raise ValueError("Inference output IDs must be non-null and unique")

    order = official_test[["id", "_test_order"]]
    merged = order.merge(merged, on="id", how="left", validate="one_to_one")
    merged = merged.sort_values("_test_order").drop(columns="_test_order").reset_index(drop=True)

    if merged["label"].isna().any() or not merged["label"].isin([0, 1]).all():
        raise ValueError("Every test row must have a binary label")

    submission = merged[["id", "label"]].copy()
    if submission["id"].tolist() != official_test["id"].tolist():
        raise ValueError("Submission order does not match the official test order")

    submission_path = OUTPUT_DIR / "submission.csv"

    submission.to_csv(submission_path, index=False)

    assert len(submission) == len(official_test)
    assert submission["id"].is_unique
    assert submission["label"].isin([0, 1]).all()
    assert not submission.isna().any().any()

    print("Wrote:", submission_path)
    print("Rows per bucket:")
    print(pd.Series({key: len(df) for key, df in bucket_dfs.items()}))
    print("Label counts:")
    print(submission["label"].value_counts().sort_index())
    print(submission.head())
    return submission


In [ ]:
# Execute the complete offline inference pipeline.
submission = run_separated_pipeline()
